In [1]:
# Create Deployment and Maintenance Infrastructure

from pathlib import Path
import sys
import os
import json
import hashlib
import platform
import subprocess
import importlib.util
from datetime import datetime, timezone

import pandas as pd
import numpy as np

# Locate project root from the notebook working directory
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError(
        f"Project directory could not be verified: {PROJECT_ROOT}"
    )

MODELS_DIR = PROJECT_ROOT / "models"
CONFIG_DIR = PROJECT_ROOT / "config"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"
MONITORING_DIR = PROJECT_ROOT / "monitoring"
LOG_DIR = MONITORING_DIR / "logs"
REGISTRY_DIR = MODELS_DIR / "registry"

for directory in [
    MODELS_DIR, CONFIG_DIR, DEPLOYMENT_DIR,
    MONITORING_DIR, LOG_DIR, REGISTRY_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

# Create development and production configurations
development_config = {
    "environment": "development",
    "debug": True,
    "model_directory": "models",
    "data_directory": "data/processed",
    "output_directory": "outputs",
    "log_directory": "monitoring/logs",
    "forecast_target": "children_in_hhs_care",
    "forecast_horizons": [1, 7, 14, 30],
    "random_seed": 42,
    "monitoring_enabled": True
}

production_config = {
    "environment": "production",
    "debug": False,
    "model_directory": "models",
    "data_directory": "data/processed",
    "output_directory": "outputs",
    "log_directory": "monitoring/logs",
    "forecast_target": "children_in_hhs_care",
    "forecast_horizons": [1, 7, 14, 30],
    "random_seed": 42,
    "monitoring_enabled": True
}

for name, config in [
    ("development", development_config),
    ("production", production_config)
]:
    (CONFIG_DIR / f"{name}.json").write_text(
        json.dumps(config, indent=4),
        encoding="utf-8"
    )

# Create model loader using an explicit project path
model_loader_code = '''
from pathlib import Path
import os
import pickle
import joblib
import json

PROJECT_ROOT = Path(__file__).resolve().parent.parent
MODEL_DIR = PROJECT_ROOT / "models"

SUPPORTED_EXTENSIONS = {".joblib", ".pkl", ".pickle"}

def discover_models(model_directory=None):
    directory = Path(model_directory or MODEL_DIR)

    if not directory.exists():
        return []

    return sorted(
        path for path in directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
        and "registry" not in path.parts
    )

def load_model(model_path=None, model_name=None):
    selected_path = model_path or os.getenv("FORECAST_MODEL_PATH")

    if selected_path:
        path = Path(selected_path)

        if not path.is_absolute():
            path = PROJECT_ROOT / path
    else:
        available = discover_models()

        if model_name:
            available = [
                item for item in available
                if item.name == model_name or item.stem == model_name
            ]

        if len(available) == 0:
            raise FileNotFoundError(
                "No supported trained model was found."
            )

        if len(available) > 1:
            raise RuntimeError(
                "Multiple model artifacts found. Specify the exact path."
            )

        path = available[0]

    if not path.exists():
        raise FileNotFoundError(f"Model does not exist: {path}")

    if path.suffix.lower() == ".joblib":
        model = joblib.load(path)
    elif path.suffix.lower() in {".pkl", ".pickle"}:
        with open(path, "rb") as file:
            model = pickle.load(file)
    else:
        raise ValueError("Unsupported model format.")

    return model

def predict_with_model(model, features):
    if not hasattr(model, "predict"):
        raise TypeError("Model does not expose predict().")

    return model.predict(features)

def load_project_config(environment=None):
    environment = (
        environment or os.getenv("APP_ENV", "development")
    ).lower()

    if environment not in {"development", "production"}:
        raise ValueError("Invalid APP_ENV.")

    config_path = PROJECT_ROOT / "config" / f"{environment}.json"

    with open(config_path, encoding="utf-8") as file:
        return json.load(file)
'''

(MODELS_DIR / "model_loader.py").write_text(
    model_loader_code, encoding="utf-8"
)

# Create monitoring utilities
monitoring_code = '''
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import json

PROJECT_ROOT = Path(__file__).resolve().parent.parent
LOG_DIR = PROJECT_ROOT / "monitoring" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

def log_forecast(actual, predicted, model_version, forecast_date=None):
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "forecast_date": str(forecast_date) if forecast_date is not None else None,
        "actual": float(actual) if actual is not None and pd.notna(actual) else None,
        "predicted": float(predicted) if predicted is not None and pd.notna(predicted) else None,
        "model_version": str(model_version)
    }

    path = LOG_DIR / "forecast_monitoring.csv"

    pd.DataFrame([record]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )

def calculate_error_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual = actual[valid]
    predicted = predicted[valid]

    if len(actual) == 0:
        raise ValueError("No valid observations.")

    error = actual - predicted
    denominator = np.abs(actual) + np.abs(predicted)
    valid_smape = denominator != 0
    nonzero = actual != 0

    return {
        "observations": int(len(actual)),
        "MAE": float(np.mean(np.abs(error))),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "MAPE_percent": (
            float(np.mean(np.abs(error[nonzero] / actual[nonzero])) * 100)
            if nonzero.any() else None
        ),
        "sMAPE_percent": (
            float(np.mean(
                2 * np.abs(error[valid_smape]) /
                denominator[valid_smape]
            ) * 100)
            if valid_smape.any() else None
        )
    }

def calculate_psi(reference, current, bins=10):
    reference = np.asarray(reference, dtype=float)
    current = np.asarray(current, dtype=float)

    reference = reference[np.isfinite(reference)]
    current = current[np.isfinite(current)]

    if len(reference) < 2 or len(current) < 2:
        raise ValueError("At least two valid values are required.")

    edges = np.unique(
        np.quantile(reference, np.linspace(0, 1, bins + 1))
    )

    if len(edges) < 3:
        return 0.0

    edges[0] = -np.inf
    edges[-1] = np.inf

    ref_counts, _ = np.histogram(reference, bins=edges)
    cur_counts, _ = np.histogram(current, bins=edges)

    ref_pct = np.maximum(ref_counts / len(reference), 0.0001)
    cur_pct = np.maximum(cur_counts / len(current), 0.0001)

    return float(np.sum(
        (cur_pct - ref_pct) * np.log(cur_pct / ref_pct)
    ))

def classify_drift(psi):
    if psi < 0.10:
        return "No material drift signal"
    elif psi < 0.25:
        return "Moderate drift signal - investigate"
    return "Significant drift signal - review model"

def evaluate_drift(reference, current):
    psi = calculate_psi(reference, current)

    return {
        "PSI": psi,
        "interpretation": classify_drift(psi),
        "timestamp_utc": datetime.now(timezone.utc).isoformat()
    }

def save_monitoring_report(report, filename="drift_report.json"):
    path = LOG_DIR / filename

    with open(path, "w", encoding="utf-8") as file:
        json.dump(report, file, indent=4)

    return path
'''

(MONITORING_DIR / "monitoring_utils.py").write_text(
    monitoring_code, encoding="utf-8"
)

# Create dashboard integration helper
dashboard_helper_code = '''
from pathlib import Path
import sys
import streamlit as st
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.model_loader import load_model, load_project_config

@st.cache_resource
def get_forecasting_model(model_path=None):
    return load_model(model_path=model_path)

def get_forecast_config():
    return load_project_config()

def generate_forecast(model, feature_dataframe):
    if not isinstance(feature_dataframe, pd.DataFrame):
        raise TypeError("Features must be a pandas DataFrame.")

    return model.predict(feature_dataframe)
'''

(DEPLOYMENT_DIR / "dashboard_model_integration.py").write_text(
    dashboard_helper_code, encoding="utf-8"
)

# Create deployment requirements
requirements = """streamlit
pandas
numpy
scikit-learn
statsmodels
joblib
plotly
matplotlib
seaborn
python-dotenv
python-docx
reportlab
"""

(PROJECT_ROOT / "requirements-deployment.txt").write_text(
    requirements, encoding="utf-8"
)

# Create environment template
env_template = """APP_ENV=development
FORECAST_MODEL_PATH=
FORECAST_DATA_PATH=data/processed/cleaned_care_data.csv
"""

(PROJECT_ROOT / ".env.example").write_text(
    env_template, encoding="utf-8"
)

# Create deployment guide without Windows escape warnings
deployment_guide = r"""
# Deployment and Maintenance Guide

## Environment Setup

Create a virtual environment:

    python -m venv .venv

Activate it in Windows PowerShell:

    .venv/Scripts/Activate.ps1

Install dependencies:

    python -m pip install --upgrade pip
    pip install -r requirements-deployment.txt

## Configuration

Development:
Set APP_ENV=development.

Production:
Set APP_ENV=production.

Production configuration must keep debug mode disabled.

## Model Loading

Store approved model artifacts in the models directory.

Supported formats:
- .joblib
- .pkl
- .pickle

Set FORECAST_MODEL_PATH to the exact approved model artifact.

Only load trusted model artifacts. Pickle and joblib deserialization
can execute code.

## Dashboard Integration

The reusable helper is available at:

deployment/dashboard_model_integration.py

The model must receive exactly the same feature names, feature order,
preprocessing, and data types used during training.

Integrate only after verifying the trained model artifact and
feature-engineering pipeline.

## Clean Environment Test

1. Create a fresh virtual environment.
2. Install requirements-deployment.txt.
3. Set APP_ENV=production.
4. Set FORECAST_MODEL_PATH to the approved model.
5. Run the deployment test notebook.
6. Start the Streamlit application.
7. Test all pages, filters, charts, predictions, and downloads.

## Performance Testing

Measure model loading and prediction time using representative
production inputs. Record the input size, model version, software
environment, and elapsed time.

## Reproducibility

Record the dataset hash, model hash, Python version, package versions,
feature definitions, preprocessing, random seed, model parameters,
forecast horizon, and training/validation periods.

## Data Refresh

1. Obtain authorized updated data.
2. Preserve the original source file.
3. Validate schema, dates, duplicates, missing values, and numeric fields.
4. Do not replace missing reporting dates with zero.
5. Run the established cleaning and feature-engineering pipeline.
6. Review distribution changes.
7. Generate and validate updated forecasts.
8. Archive data and outputs with version information.

## Model Monitoring

Monitor MAE, RMSE, MAPE and sMAPE where appropriate, interval coverage,
invalid inputs, missing features, forecast failures, and latency.

Record the model version used for every forecast.

## Drift Monitoring

PSI screening thresholds:

- Below 0.10: no material drift signal.
- 0.10 to below 0.25: investigate.
- 0.25 or above: significant drift signal; review model.

These are screening thresholds, not proof of model degradation.
Investigate feature-level drift and forecast performance.

## Retraining Criteria

Review retraining when:
- Forecast error exceeds an approved threshold.
- Drift persists across monitoring periods.
- Data distributions change materially.
- Source reporting definitions change.
- Prediction interval coverage deteriorates.
- Scheduled model review identifies a need.

Retraining requires chronological validation, leakage checks,
comparison with the incumbent model, and documented approval.

## Model Versioning

Maintain a version identifier, artifact hash, dataset hash, feature schema,
model parameters, validation metrics, training timestamp, approval status,
and rollback reference.

## Security

Restrict access to data and artifacts. Keep credentials outside source code.
Use authorized data sources. Protect sensitive information in logs.

## Production Acceptance

Production readiness requires a verified trained model, matching
feature pipeline, clean-environment test, dashboard integration,
performance test, reproducibility test, monitoring, and rollback procedure.
"""

(DEPLOYMENT_DIR / "DEPLOYMENT_AND_MAINTENANCE.md").write_text(
    deployment_guide.strip() + "\n",
    encoding="utf-8"
)

# Register discovered model artifacts without assuming approval
def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

model_artifacts = []

for path in MODELS_DIR.rglob("*"):
    if (
        path.is_file()
        and path.suffix.lower() in {".joblib", ".pkl", ".pickle"}
        and "registry" not in path.parts
    ):
        model_artifacts.append({
            "model_file": str(path.relative_to(PROJECT_ROOT)),
            "version": "unassigned",
            "sha256": sha256_file(path),
            "file_size_bytes": path.stat().st_size,
            "registered_at_utc": datetime.now(timezone.utc).isoformat(),
            "status": "discovered_not_approved",
            "training_metadata": "Requires verification"
        })

registry = {
    "project": "Predictive Forecasting of Care Load & Placement Demand",
    "registry_created_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "models": model_artifacts
}

(REGISTRY_DIR / "model_registry.json").write_text(
    json.dumps(registry, indent=4),
    encoding="utf-8"
)

# Verify all generated infrastructure files
created_files = [
    CONFIG_DIR / "development.json",
    CONFIG_DIR / "production.json",
    MODELS_DIR / "model_loader.py",
    MONITORING_DIR / "monitoring_utils.py",
    DEPLOYMENT_DIR / "dashboard_model_integration.py",
    DEPLOYMENT_DIR / "DEPLOYMENT_AND_MAINTENANCE.md",
    PROJECT_ROOT / "requirements-deployment.txt",
    PROJECT_ROOT / ".env.example",
    REGISTRY_DIR / "model_registry.json"
]

print("=" * 70)
print("DEPLOYMENT INFRASTRUCTURE CREATED")
print("=" * 70)

for path in created_files:
    print(
        ("SUCCESS" if path.exists() else "FAILED"),
        path
    )

print("\nDiscovered trained model artifacts:", len(model_artifacts))

for item in model_artifacts:
    print("-", item["model_file"], "|", item["status"])

print("=" * 70)

Project root: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand


DEPLOYMENT INFRASTRUCTURE CREATED
SUCCESS C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\config\development.json
SUCCESS C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\config\production.json
SUCCESS C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\models\model_loader.py
SUCCESS C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\monitoring\monitoring_utils.py
SUCCESS C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\dashboard_model_integration.py
SUCCESS C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\DEPLOYMENT_AND_MAINTENANCE.md
SUCCESS C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\requirements-deployment.txt
SUCCESS C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\.env.example
SUCCESS C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\models\registry\model_registry.json

Discovered trained model arti

In [2]:

# Deployment Infrastructure Testing and Validation
# Complete validation with feature-name-aware model predictions

from pathlib import Path
import sys
import json
import time
import importlib.util
import importlib.metadata
import numpy as np
import pandas as pd
import joblib

# ------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError(
        f"Project root could not be verified: {PROJECT_ROOT}"
    )

print("=" * 75)
print("DEPLOYMENT INFRASTRUCTURE TESTING")
print("=" * 75)
print("Project root:", PROJECT_ROOT)

results = []

def record_test(name, passed, details=""):
    results.append({
        "Test": name,
        "Status": "PASSED" if passed else "FAILED",
        "Details": str(details)
    })

    print(
        f"{'PASS' if passed else 'FAIL'} | {name}"
        + (f" | {details}" if details else "")
    )


# ------------------------------------------------------------
# Validate infrastructure files
# ------------------------------------------------------------

required_files = [
    "config/development.json",
    "config/production.json",
    "models/model_loader.py",
    "monitoring/monitoring_utils.py",
    "deployment/dashboard_model_integration.py",
    "deployment/DEPLOYMENT_AND_MAINTENANCE.md",
    "requirements-deployment.txt",
    ".env.example",
    "models/registry/model_registry.json"
]

for relative_path in required_files:
    file_path = PROJECT_ROOT / relative_path

    record_test(
        f"File exists: {relative_path}",
        file_path.is_file(),
        f"{file_path.stat().st_size} bytes"
        if file_path.is_file() else "Missing"
    )


# ------------------------------------------------------------
# Validate JSON configurations and model registry
# ------------------------------------------------------------

for relative_path in [
    "config/development.json",
    "config/production.json",
    "models/registry/model_registry.json"
]:
    try:
        with open(
            PROJECT_ROOT / relative_path,
            "r",
            encoding="utf-8"
        ) as file:
            json_data = json.load(file)

        record_test(
            f"Valid JSON: {relative_path}",
            isinstance(json_data, dict),
            f"{len(json_data)} top-level entries"
        )

    except Exception as exc:
        record_test(
            f"Valid JSON: {relative_path}",
            False,
            str(exc)
        )


# ------------------------------------------------------------
# Discover trained model artifacts
# ------------------------------------------------------------

model_files = (
    list((PROJECT_ROOT / "models").rglob("*.joblib"))
    + list((PROJECT_ROOT / "models").rglob("*.pkl"))
    + list((PROJECT_ROOT / "models").rglob("*.pickle"))
)

model_files = sorted(set(model_files))

record_test(
    "Trained model artifacts discovered",
    len(model_files) > 0,
    f"{len(model_files)} artifacts"
)


# ------------------------------------------------------------
# Test model artifact loading
# ------------------------------------------------------------

loaded_models = []
failed_models = []

for model_path in model_files:
    try:
        model = joblib.load(model_path)
        loaded_models.append((model_path, model))

    except Exception as exc:
        failed_models.append((model_path.name, str(exc)))

record_test(
    "Model artifact loading",
    len(model_files) > 0 and len(failed_models) == 0,
    f"Loaded: {len(loaded_models)}, Failed: {len(failed_models)}"
)

for name, error in failed_models:
    print(f"MODEL LOAD ERROR: {name} | {error}")


# ------------------------------------------------------------
# Test monitoring utility import
# ------------------------------------------------------------

try:
    monitoring_path = (
        PROJECT_ROOT / "monitoring" / "monitoring_utils.py"
    )

    spec = importlib.util.spec_from_file_location(
        "deployment_monitoring_utils",
        monitoring_path
    )

    monitoring_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(monitoring_module)

    record_test("Monitoring utility import", True)

except Exception as exc:
    record_test("Monitoring utility import", False, str(exc))


# ------------------------------------------------------------
# Test model loader import
# ------------------------------------------------------------

try:
    loader_path = PROJECT_ROOT / "models" / "model_loader.py"

    spec = importlib.util.spec_from_file_location(
        "deployment_model_loader",
        loader_path
    )

    loader_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(loader_module)

    record_test("Model loader import", True)

except Exception as exc:
    record_test("Model loader import", False, str(exc))


# ------------------------------------------------------------
# Feature-name-aware prediction reproducibility test
# ------------------------------------------------------------

prediction_test_passed = False
performance_test_passed = False

for model_path, model in loaded_models:

    if not hasattr(model, "predict"):
        continue

    try:
        # Prefer the exact feature names used during model training.

        if hasattr(model, "feature_names_in_"):

            feature_names = list(model.feature_names_in_)

            sample = pd.DataFrame(
                np.zeros((1, len(feature_names))),
                columns=feature_names
            )

        elif hasattr(model, "n_features_in_"):

            feature_count = int(model.n_features_in_)

            sample = pd.DataFrame(
                np.zeros((1, feature_count)),
                columns=[
                    f"feature_{i}"
                    for i in range(feature_count)
                ]
            )

        else:
            print(
                f"Prediction test skipped: {model_path.name} "
                "- no identifiable feature schema."
            )
            continue

        # Execute prediction and measure runtime.

        start_time = time.perf_counter()

        prediction_1 = np.asarray(model.predict(sample))

        execution_time = time.perf_counter() - start_time

        prediction_2 = np.asarray(model.predict(sample))

        reproducible = (
            prediction_1.shape == prediction_2.shape
            and np.allclose(
                prediction_1,
                prediction_2,
                equal_nan=True
            )
        )

        record_test(
            f"Prediction reproducibility: {model_path.name}",
            reproducible
        )

        prediction_test_passed = reproducible

        valid_runtime = (
            np.isfinite(execution_time)
            and execution_time >= 0
        )

        record_test(
            f"Prediction execution: {model_path.name}",
            valid_runtime,
            f"{execution_time:.6f} seconds"
        )

        performance_test_passed = valid_runtime

        # One compatible model is sufficient for this basic smoke test.
        break

    except Exception as exc:
        print(
            f"Prediction test skipped for "
            f"{model_path.name}: {exc}"
        )


if not prediction_test_passed:
    record_test(
        "Prediction reproducibility",
        False,
        "No compatible model produced a reproducible prediction."
    )

if not performance_test_passed:
    record_test(
        "Prediction performance",
        False,
        "No compatible model completed the prediction test."
    )


# ------------------------------------------------------------
# Validate deployment dependencies
# ------------------------------------------------------------

required_packages = [
    "streamlit",
    "pandas",
    "numpy",
    "scikit-learn",
    "statsmodels",
    "joblib",
    "plotly"
]

for package in required_packages:

    try:
        version = importlib.metadata.version(package)

        record_test(
            f"Dependency available: {package}",
            True,
            f"Version {version}"
        )

    except importlib.metadata.PackageNotFoundError:

        record_test(
            f"Dependency available: {package}",
            False,
            "Package not installed"
        )


# ------------------------------------------------------------
# Generate deployment validation report
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

output_dir = PROJECT_ROOT / "outputs" / "deployment"
output_dir.mkdir(parents=True, exist_ok=True)

report_path = output_dir / "deployment_validation_report.csv"

results_df.to_csv(report_path, index=False)

passed = int(
    (results_df["Status"] == "PASSED").sum()
)

failed = int(
    (results_df["Status"] == "FAILED").sum()
)

total = len(results_df)

pass_percentage = (
    round((passed / total) * 100, 2)
    if total else 0
)

print("\n" + "=" * 75)
print("DEPLOYMENT VALIDATION SUMMARY")
print("=" * 75)

display(results_df)

print("\nTotal tests:", total)
print("Passed:", passed)
print("Failed:", failed)
print("Pass percentage:", pass_percentage, "%")

print("\nValidation report saved:")
print(report_path)

print("=" * 75)


DEPLOYMENT INFRASTRUCTURE TESTING
Project root: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand
PASS | File exists: config/development.json | 400 bytes
PASS | File exists: config/production.json | 400 bytes
PASS | File exists: models/model_loader.py | 2393 bytes
PASS | File exists: monitoring/monitoring_utils.py | 3533 bytes
PASS | File exists: deployment/dashboard_model_integration.py | 691 bytes
PASS | File exists: deployment/DEPLOYMENT_AND_MAINTENANCE.md | 3842 bytes
PASS | File exists: requirements-deployment.txt | 129 bytes
PASS | File exists: .env.example | 100 bytes
PASS | File exists: models/registry/model_registry.json | 9410 bytes
PASS | Valid JSON: config/development.json | 10 top-level entries
PASS | Valid JSON: config/production.json | 10 top-level entries
PASS | Valid JSON: models/registry/model_registry.json | 5 top-level entries
PASS | Trained model artifacts discovered | 20 artifacts


PASS | Model artifact loading | Loaded: 20, Failed: 0
PASS | Monitoring utility import
PASS | Model loader import
PASS | Prediction reproducibility: gradient_boosting_14_day.joblib
PASS | Prediction execution: gradient_boosting_14_day.joblib | 0.001661 seconds
PASS | Dependency available: streamlit | Version 1.66.1
PASS | Dependency available: pandas | Version 2.2.3
PASS | Dependency available: numpy | Version 1.26.4
PASS | Dependency available: scikit-learn | Version 1.4.2
PASS | Dependency available: statsmodels | Version 0.14.2
PASS | Dependency available: joblib | Version 1.4.2
PASS | Dependency available: plotly | Version 7.1.0

DEPLOYMENT VALIDATION SUMMARY


,Test,Status,Details
0,File exists: config/development.json,PASSED,400 bytes
1,File exists: config/production.json,PASSED,400 bytes
2,File exists: models/model_loader.py,PASSED,2393 bytes
3,File exists: monitoring/monitoring_utils.py,PASSED,3533 bytes
4,File exists: deployment/dashboard_model_integr...,PASSED,691 bytes
5,File exists: deployment/DEPLOYMENT_AND_MAINTEN...,PASSED,3842 bytes
6,File exists: requirements-deployment.txt,PASSED,129 bytes
7,File exists: .env.example,PASSED,100 bytes
8,File exists: models/registry/model_registry.json,PASSED,9410 bytes
9,Valid JSON: config/development.json,PASSED,10 top-level entries



Total tests: 25
Passed: 25
Failed: 0
Pass percentage: 100.0 %

Validation report saved:
C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\deployment_validation_report.csv


In [3]:

# Complete Deployment, Integration, Testing and Maintenance
# Covers deployment checklist items 16.46 to 16.56

from pathlib import Path
import os
import sys
import json
import time
import shutil
import hashlib
import subprocess
import importlib
import importlib.metadata
import importlib.util
import traceback
import warnings

import numpy as np
import pandas as pd
import joblib

# ------------------------------------------------------------
# Project configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

APP_PATH = PROJECT_ROOT / "app.py"
MODEL_DIR = PROJECT_ROOT / "models"
REGISTRY_PATH = MODEL_DIR / "registry" / "model_registry.json"

DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"
MONITORING_DIR = PROJECT_ROOT / "monitoring"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deployment"
BACKUP_DIR = PROJECT_ROOT / "backups" / "deployment"

for folder in [
    DEPLOYMENT_DIR,
    MONITORING_DIR,
    OUTPUT_DIR,
    BACKUP_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("COMPLETE DEPLOYMENT AND MAINTENANCE VALIDATION")
print("=" * 75)
print("Project:", PROJECT_ROOT)

results = []

def add_result(task, status, details=""):
    results.append({
        "Task": task,
        "Status": status,
        "Details": str(details)
    })
    print(f"{status} | {task} | {details}")


# ------------------------------------------------------------
# 16.46 - Inspect and prepare actual dashboard integration
# ------------------------------------------------------------

if APP_PATH.is_file():

    app_source = APP_PATH.read_text(
        encoding="utf-8",
        errors="replace"
    )

    app_hash_before = hashlib.sha256(
        app_source.encode("utf-8")
    ).hexdigest()

    backup_path = BACKUP_DIR / (
        f"app_backup_{time.strftime('%Y%m%d_%H%M%S')}.py"
    )

    shutil.copy2(APP_PATH, backup_path)

    dashboard_sections = [
        "historical",
        "forecast",
        "capacity",
        "scenario",
        "download"
    ]

    detected_sections = [
        section for section in dashboard_sections
        if section in app_source.lower()
    ]

    integration_source = r'''
"""Safe model deployment integration utilities."""

from pathlib import Path
import json
import joblib
import streamlit as st

PROJECT_ROOT = Path(__file__).resolve().parents[1]
REGISTRY_PATH = PROJECT_ROOT / "models" / "registry" / "model_registry.json"

def load_registry():
    if not REGISTRY_PATH.exists():
        return {}
    with open(REGISTRY_PATH, "r", encoding="utf-8") as file:
        return json.load(file)

def get_model_artifacts():
    registry = load_registry()
    artifacts = registry.get("models", [])
    if isinstance(artifacts, dict):
        artifacts = list(artifacts.values())
    return artifacts

def render_model_deployment_status():
    st.subheader("Model Deployment Status")

    try:
        artifacts = get_model_artifacts()
        st.write("Registered model artifacts:", len(artifacts))

        if artifacts:
            st.dataframe(artifacts, use_container_width=True)
        else:
            st.info("No registered models are available.")

    except Exception as exc:
        st.error(f"Could not read model registry: {exc}")

def load_trusted_model(model_path):
    path = Path(model_path).resolve()

    if not path.is_relative_to(PROJECT_ROOT.resolve()):
        raise ValueError("Model path must be inside the project directory.")

    if not path.is_file():
        raise FileNotFoundError(path)

    return joblib.load(path)
'''

    integration_path = (
        DEPLOYMENT_DIR / "dashboard_model_integration.py"
    )

    integration_path.write_text(
        integration_source,
        encoding="utf-8"
    )

    # Do not automatically inject model predictions into an unknown
    # dashboard: its real feature preparation must be preserved.

    app_hash_after = hashlib.sha256(
        APP_PATH.read_bytes()
    ).hexdigest()

    add_result(
        "16.46 Dashboard inspection and integration helper",
        "PASSED",
        f"app.py inspected; backup: {backup_path.name}; "
        f"detected keywords: {detected_sections}; "
        f"app unchanged: {app_hash_before == app_hash_after}. "
        "Final prediction integration requires matching real input features."
    )

else:
    add_result(
        "16.46 Dashboard inspection and integration helper",
        "FAILED",
        "app.py not found at project root."
    )


# ------------------------------------------------------------
# 16.47 - Generate pinned deployment dependencies
# ------------------------------------------------------------

packages = [
    "streamlit",
    "pandas",
    "numpy",
    "scikit-learn",
    "statsmodels",
    "joblib",
    "plotly"
]

dependency_lines = []
dependency_errors = []

for package in packages:
    try:
        version = importlib.metadata.version(package)
        dependency_lines.append(f"{package}=={version}")
    except importlib.metadata.PackageNotFoundError:
        dependency_errors.append(package)

lock_path = DEPLOYMENT_DIR / "requirements-deployment-lock.txt"

lock_path.write_text(
    "\n".join(dependency_lines) + "\n",
    encoding="utf-8"
)

add_result(
    "16.47 Pinned deployment dependencies",
    "PASSED" if not dependency_errors else "WARNING",
    f"Locked {len(dependency_lines)} packages; "
    f"missing: {dependency_errors}"
)


# ------------------------------------------------------------
# 16.48 - Generate deployment instructions
# ------------------------------------------------------------

guide = r"""
# Deployment Instructions

## Prerequisites
- Python version compatible with the locked dependencies.
- Access to the project data and trained model artifacts.

## Installation (Windows PowerShell)

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install --upgrade pip
pip install -r deployment\requirements-deployment-lock.txt
```

## Run dashboard

```powershell
streamlit run app.py
```

## Production configuration
Use config/production.json and configure environment-specific values.
Do not commit secrets or credentials.

## Model loading
Use models/model_loader.py and the approved model registry.
Only load trusted model artifacts.

## Data refresh
1. Obtain the latest source data.
2. Validate column names, data types, dates and missingness.
3. Preserve the original source file.
4. Run the approved preprocessing pipeline.
5. Save the refreshed processed dataset.
6. Record refresh time and data version.
7. Revalidate forecasts before deployment.

## Monitoring
Monitor prediction errors, missing inputs, forecast deviations,
data drift, application errors and prediction latency.

## Retraining
Retrain when performance breaches approved thresholds,
data definitions change, or sustained data drift is detected.

## Rollback
Keep the previous approved model artifact and application backup.
Never replace a production model without validation.
"""

guide_path = DEPLOYMENT_DIR / "PRODUCTION_DEPLOYMENT_GUIDE.md"

guide_path.write_text(guide, encoding="utf-8")

add_result(
    "16.48 Deployment instructions",
    "PASSED",
    str(guide_path)
)


# ------------------------------------------------------------
# 16.49 - Prepare isolated clean-environment test
# ------------------------------------------------------------

clean_test_script = r'''
import importlib.metadata
import sys

packages = [
    "streamlit",
    "pandas",
    "numpy",
    "scikit-learn",
    "statsmodels",
    "joblib",
    "plotly"
]

failed = []

for package in packages:
    try:
        print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        failed.append(package)

if failed:
    print("MISSING:", failed)
    sys.exit(1)

print("CLEAN ENVIRONMENT DEPENDENCY CHECK PASSED")
'''

clean_script_path = DEPLOYMENT_DIR / "clean_environment_test.py"

clean_script_path.write_text(
    clean_test_script,
    encoding="utf-8"
)

clean_venv_path = PROJECT_ROOT / ".deployment_test_env"

add_result(
    "16.49 Clean environment test preparation",
    "PREPARED",
    f"Isolated environment script: {clean_script_path}; "
    f"venv location reserved: {clean_venv_path}. "
    "Actual isolated installation and execution still required."
)


# ------------------------------------------------------------
# 16.50 - Benchmark real model prediction execution
# ------------------------------------------------------------

model_files = sorted(
    list(MODEL_DIR.rglob("*.joblib"))
    + list(MODEL_DIR.rglob("*.pkl"))
    + list(MODEL_DIR.rglob("*.pickle"))
)

benchmark_results = []

for model_path in model_files:

    try:
        model = joblib.load(model_path)

        if not hasattr(model, "predict"):
            continue

        if hasattr(model, "feature_names_in_"):
            feature_names = list(model.feature_names_in_)
            sample = pd.DataFrame(
                np.zeros((1, len(feature_names))),
                columns=feature_names
            )

        elif hasattr(model, "n_features_in_"):
            feature_count = int(model.n_features_in_)
            sample = pd.DataFrame(
                np.zeros((1, feature_count)),
                columns=[
                    f"feature_{i}"
                    for i in range(feature_count)
                ]
            )

        else:
            continue

        start = time.perf_counter()
        prediction = model.predict(sample)
        elapsed = time.perf_counter() - start

        benchmark_results.append({
            "model": str(model_path.relative_to(PROJECT_ROOT)),
            "execution_seconds": elapsed,
            "prediction_shape": str(np.asarray(prediction).shape),
            "status": "PASSED"
        })

    except Exception as exc:
        benchmark_results.append({
            "model": str(model_path.relative_to(PROJECT_ROOT)),
            "execution_seconds": None,
            "prediction_shape": None,
            "status": "FAILED",
            "error": str(exc)
        })

benchmark_df = pd.DataFrame(benchmark_results)

benchmark_path = OUTPUT_DIR / "model_performance_benchmark.csv"

benchmark_df.to_csv(benchmark_path, index=False)

successful_benchmarks = (
    int((benchmark_df["status"] == "PASSED").sum())
    if not benchmark_df.empty else 0
)

add_result(
    "16.50 Model inference performance benchmark",
    "PASSED" if successful_benchmarks > 0 else "WARNING",
    f"{successful_benchmarks}/{len(benchmark_df)} model tests passed. "
    "Synthetic-input timings are not dashboard load-time benchmarks."
)


# ------------------------------------------------------------
# 16.51 - Test prediction reproducibility
# ------------------------------------------------------------

reproducibility_results = []

for model_path in model_files:

    try:
        model = joblib.load(model_path)

        if not hasattr(model, "predict"):
            continue

        if hasattr(model, "feature_names_in_"):
            names = list(model.feature_names_in_)
            sample = pd.DataFrame(
                np.zeros((1, len(names))),
                columns=names
            )

        elif hasattr(model, "n_features_in_"):
            count = int(model.n_features_in_)
            sample = pd.DataFrame(
                np.zeros((1, count)),
                columns=[f"feature_{i}" for i in range(count)]
            )

        else:
            continue

        first = np.asarray(model.predict(sample))
        second = np.asarray(model.predict(sample))

        reproducible = (
            first.shape == second.shape
            and np.allclose(first, second, equal_nan=True)
        )

        reproducibility_results.append({
            "model": model_path.name,
            "reproducible": bool(reproducible)
        })

    except Exception as exc:
        reproducibility_results.append({
            "model": model_path.name,
            "reproducible": False,
            "error": str(exc)
        })

repro_df = pd.DataFrame(reproducibility_results)

repro_path = OUTPUT_DIR / "model_reproducibility_report.csv"

repro_df.to_csv(repro_path, index=False)

repro_pass = (
    len(repro_df) > 0
    and bool(repro_df["reproducible"].all())
)

add_result(
    "16.51 Prediction reproducibility",
    "PASSED" if repro_pass else "WARNING",
    f"{int(repro_df['reproducible'].sum()) if len(repro_df) else 0}/"
    f"{len(repro_df)} models reproducible on repeated synthetic inputs."
)


# ------------------------------------------------------------
# 16.52 - Create controlled data refresh procedure
# ------------------------------------------------------------

refresh_script = r'''
from pathlib import Path
from datetime import datetime, timezone
import shutil
import pandas as pd
import json

PROJECT_ROOT = Path(__file__).resolve().parents[1]

SOURCE = Path(
    input("Enter the new source CSV path: ").strip().strip('"')
)

DESTINATION = (
    PROJECT_ROOT / "data" / "processed" / "cleaned_care_data.csv"
)

if not SOURCE.is_file():
    raise FileNotFoundError(SOURCE)

new_data = pd.read_csv(SOURCE)

required_columns = [
    "date",
    "children_apprehended_and_placed_in_cbp_custody",
    "children_in_cbp_custody",
    "children_transferred_out_of_cbp_custody",
    "children_in_hhs_care",
    "children_discharged_from_hhs_care"
]

missing = [
    col for col in required_columns
    if col not in new_data.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

new_data["date"] = pd.to_datetime(
    new_data["date"],
    errors="raise"
)

if new_data["date"].duplicated().any():
    raise ValueError("Duplicate dates detected.")

numeric_columns = [
    col for col in required_columns if col != "date"
]

for col in numeric_columns:
    new_data[col] = pd.to_numeric(
        new_data[col],
        errors="raise"
    )

if (new_data[numeric_columns] < 0).any().any():
    raise ValueError("Negative counts detected.")

backup_dir = PROJECT_ROOT / "backups" / "data_refresh"
backup_dir.mkdir(parents=True, exist_ok=True)

if DESTINATION.exists():
    backup_name = (
        "cleaned_care_data_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
        + ".csv"
    )
    shutil.copy2(
        DESTINATION,
        backup_dir / backup_name
    )

new_data = new_data.sort_values("date")

new_data.to_csv(DESTINATION, index=False)

metadata = {
    "refresh_time_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": str(SOURCE),
    "rows": len(new_data),
    "first_date": str(new_data["date"].min()),
    "last_date": str(new_data["date"].max())
}

metadata_path = (
    PROJECT_ROOT / "data" / "processed" / "refresh_metadata.json"
)

metadata_path.write_text(
    json.dumps(metadata, indent=4),
    encoding="utf-8"
)

print("Data refresh completed:", DESTINATION)
print(metadata)
'''

refresh_path = DEPLOYMENT_DIR / "refresh_data.py"

refresh_path.write_text(
    refresh_script,
    encoding="utf-8"
)

add_result(
    "16.52 Data refresh procedure",
    "PASSED",
    f"Controlled refresh script created: {refresh_path}. "
    "It validates and backs up data before replacement."
)


# ------------------------------------------------------------
# 16.53 and 16.54 - Monitoring and drift detection
# ------------------------------------------------------------

monitoring_source = r'''
"""Model performance and drift monitoring utilities."""

import numpy as np
import pandas as pd

def calculate_regression_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    if actual.shape != predicted.shape:
        raise ValueError("Actual and predicted shapes must match.")

    if len(actual) == 0:
        raise ValueError("No observations supplied.")

    error = actual - predicted

    return {
        "MAE": float(np.mean(np.abs(error))),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "Bias": float(np.mean(error))
    }

def calculate_psi(reference, current, bins=10):
    reference = np.asarray(reference, dtype=float)
    current = np.asarray(current, dtype=float)

    reference = reference[np.isfinite(reference)]
    current = current[np.isfinite(current)]

    if len(reference) == 0 or len(current) == 0:
        raise ValueError("Both samples must contain valid values.")

    if np.all(reference == reference[0]):
        return 0.0 if np.all(current == current[0]) else float("inf")

    edges = np.unique(
        np.quantile(reference, np.linspace(0, 1, bins + 1))
    )

    if len(edges) < 2:
        return 0.0

    edges[0] = -np.inf
    edges[-1] = np.inf

    ref_counts = np.histogram(reference, bins=edges)[0]
    cur_counts = np.histogram(current, bins=edges)[0]

    ref_pct = ref_counts / len(reference)
    cur_pct = cur_counts / len(current)

    epsilon = 1e-6
    ref_pct = np.clip(ref_pct, epsilon, None)
    cur_pct = np.clip(cur_pct, epsilon, None)

    return float(
        np.sum(
            (cur_pct - ref_pct)
            * np.log(cur_pct / ref_pct)
        )
    )

def classify_psi(psi):
    if psi < 0.10:
        return "LOW"
    if psi < 0.25:
        return "MODERATE"
    return "HIGH"

def monitoring_report(reference, current):
    psi = calculate_psi(reference, current)

    return {
        "PSI": psi,
        "drift_level": classify_psi(psi)
    }
'''

monitoring_path = MONITORING_DIR / "production_monitoring.py"

monitoring_path.write_text(
    monitoring_source,
    encoding="utf-8"
)

spec = importlib.util.spec_from_file_location(
    "production_monitoring",
    monitoring_path
)

monitoring_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(monitoring_module)

metric_test = monitoring_module.calculate_regression_metrics(
    [10, 20, 30],
    [11, 19, 30]
)

drift_test = monitoring_module.monitoring_report(
    np.arange(100, dtype=float),
    np.arange(100, dtype=float) + 1
)

add_result(
    "16.53 Model performance monitoring",
    "PASSED" if metric_test["MAE"] >= 0 else "FAILED",
    str(metric_test)
)

add_result(
    "16.54 Model drift monitoring",
    "PASSED" if "PSI" in drift_test else "FAILED",
    str(drift_test)
)


# ------------------------------------------------------------
# 16.55 - Define retraining criteria
# ------------------------------------------------------------

retraining_policy = {
    "policy_version": "1.0",
    "review_frequency": "Monthly",
    "retraining_triggers": [
        "PSI >= 0.25 for an important model input",
        "MAE exceeds the approved validation baseline by 20 percent",
        "RMSE exceeds the approved validation baseline by 20 percent",
        "Major source data definition change",
        "Sustained forecast bias",
        "Model artifact or dependency incompatibility"
    ],
    "approval_requirements": [
        "Re-run time-aware validation",
        "Compare against the incumbent model",
        "Review prediction intervals and forecast bias",
        "Document model version and training data",
        "Obtain approval before production replacement"
    ],
    "warning": (
        "Thresholds are proposed operational defaults. "
        "Calibrate them using approved historical validation results."
    )
}

retraining_path = (
    DEPLOYMENT_DIR / "retraining_policy.json"
)

retraining_path.write_text(
    json.dumps(retraining_policy, indent=4),
    encoding="utf-8"
)

add_result(
    "16.55 Retraining criteria",
    "PASSED",
    str(retraining_path)
)


# ------------------------------------------------------------
# 16.56 - Model version documentation
# ------------------------------------------------------------

version_records = []

for model_path in model_files:

    try:
        digest = hashlib.sha256(
            model_path.read_bytes()
        ).hexdigest()

        version_records.append({
            "model_name": model_path.stem,
            "relative_path": str(
                model_path.relative_to(PROJECT_ROOT)
            ),
            "artifact_version": "unassigned",
            "sha256": digest,
            "approval_status": "discovered_not_approved",
            "documentation_status": "hash_recorded_version_metadata_required"
        })

    except Exception as exc:
        version_records.append({
            "model_name": model_path.stem,
            "relative_path": str(
                model_path.relative_to(PROJECT_ROOT)
            ),
            "approval_status": "documentation_error",
            "error": str(exc)
        })

version_path = (
    MODEL_DIR / "registry" / "model_version_inventory.json"
)

version_path.parent.mkdir(parents=True, exist_ok=True)

version_path.write_text(
    json.dumps(version_records, indent=4),
    encoding="utf-8"
)

add_result(
    "16.56 Model version inventory",
    "PASSED" if len(version_records) > 0 else "FAILED",
    f"{len(version_records)} artifacts documented with SHA-256 hashes. "
    "Training metadata and approval remain to be assigned."
)


# ------------------------------------------------------------
# Final deployment checklist and reports
# ------------------------------------------------------------

final_df = pd.DataFrame(results)

final_report_path = (
    OUTPUT_DIR / "complete_deployment_checklist.csv"
)

final_df.to_csv(final_report_path, index=False)

summary = {
    "project_root": str(PROJECT_ROOT),
    "execution_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "total_checks": len(final_df),
    "passed": int((final_df["Status"] == "PASSED").sum()),
    "prepared": int((final_df["Status"] == "PREPARED").sum()),
    "warnings": int((final_df["Status"] == "WARNING").sum()),
    "failed": int((final_df["Status"] == "FAILED").sum()),
    "dashboard_file_preserved": APP_PATH.exists(),
    "dashboard_integration_requires_feature_schema": True
}

summary_path = OUTPUT_DIR / "deployment_summary.json"

summary_path.write_text(
    json.dumps(summary, indent=4),
    encoding="utf-8"
)

print("\n" + "=" * 75)
print("FINAL DEPLOYMENT CHECKLIST")
print("=" * 75)

display(final_df)

print("\nSUMMARY")
print(json.dumps(summary, indent=4))

print("\nGenerated reports:")
print(final_report_path)
print(summary_path)

print("\nAdditional artifacts:")
print(guide_path)
print(lock_path)
print(clean_script_path)
print(refresh_path)
print(monitoring_path)
print(retraining_path)
print(version_path)

print("\nIMPORTANT:")
print("Dashboard model inference integration still requires")
print("the existing dashboard's actual feature preparation.")
print("Clean-environment installation must be executed separately.")
print("Synthetic model timings are not full Streamlit benchmarks.")

print("=" * 75)


COMPLETE DEPLOYMENT AND MAINTENANCE VALIDATION
Project: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand
PASSED | 16.46 Dashboard inspection and integration helper | app.py inspected; backup: app_backup_20261105_175659.py; detected keywords: ['historical', 'forecast', 'capacity', 'scenario', 'download']; app unchanged: False. Final prediction integration requires matching real input features.
PASSED | 16.47 Pinned deployment dependencies | Locked 7 packages; missing: []
PASSED | 16.48 Deployment instructions | C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\PRODUCTION_DEPLOYMENT_GUIDE.md
PREPARED | 16.49 Clean environment test preparation | Isolated environment script: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\clean_environment_test.py; venv location reserved: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\.deployment_test_env. Actual isolated installation and execution still require

PASSED | 16.50 Model inference performance benchmark | 10/10 model tests passed. Synthetic-input timings are not dashboard load-time benchmarks.


PASSED | 16.51 Prediction reproducibility | 10/10 models reproducible on repeated synthetic inputs.
PASSED | 16.52 Data refresh procedure | Controlled refresh script created: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\refresh_data.py. It validates and backs up data before replacement.
PASSED | 16.53 Model performance monitoring | {'MAE': 0.6666666666666666, 'RMSE': 0.816496580927726, 'Bias': 0.0}
PASSED | 16.54 Model drift monitoring | {'PSI': 0.002006706954621512, 'drift_level': 'LOW'}
PASSED | 16.55 Retraining criteria | C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\retraining_policy.json
PASSED | 16.56 Model version inventory | 20 artifacts documented with SHA-256 hashes. Training metadata and approval remain to be assigned.

FINAL DEPLOYMENT CHECKLIST


,Task,Status,Details
0,16.46 Dashboard inspection and integration helper,PASSED,app.py inspected; backup: app_backup_20261105_...
1,16.47 Pinned deployment dependencies,PASSED,Locked 7 packages; missing: []
2,16.48 Deployment instructions,PASSED,C:\Users\hp\Predictive Forecasting of Care Loa...
3,16.49 Clean environment test preparation,PREPARED,Isolated environment script: C:\Users\hp\Predi...
4,16.50 Model inference performance benchmark,PASSED,10/10 model tests passed. Synthetic-input timi...
5,16.51 Prediction reproducibility,PASSED,10/10 models reproducible on repeated syntheti...
6,16.52 Data refresh procedure,PASSED,Controlled refresh script created: C:\Users\hp...
7,16.53 Model performance monitoring,PASSED,"{'MAE': 0.6666666666666666, 'RMSE': 0.81649658..."
8,16.54 Model drift monitoring,PASSED,"{'PSI': 0.002006706954621512, 'drift_level': '..."
9,16.55 Retraining criteria,PASSED,C:\Users\hp\Predictive Forecasting of Care Loa...



SUMMARY
{
    "project_root": "C:\\Users\\hp\\Predictive Forecasting of Care Load & Placement Demand",
    "execution_time": "2026-11-05 17:57:00",
    "total_checks": 11,
    "passed": 10,
    "prepared": 1,
    "warnings": 0,
    "failed": 0,
    "dashboard_file_preserved": true,
    "dashboard_integration_requires_feature_schema": true
}

Generated reports:
C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\complete_deployment_checklist.csv
C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\deployment_summary.json

Additional artifacts:
C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\PRODUCTION_DEPLOYMENT_GUIDE.md
C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\requirements-deployment-lock.txt
C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\clean_environment_test.py
C:\Users\hp\Predictive Forecasting of Care Load & Placement

In [4]:

from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

ENVIRONMENTS = [
    PROJECT_ROOT / ".deployment_test_env",
    PROJECT_ROOT / ".venv_deployment"
]

TEST_SCRIPT = PROJECT_ROOT / "deployment" / "clean_environment_test.py"

print("=" * 75)
print("FINAL CLEAN DEPLOYMENT ENVIRONMENT VALIDATION")
print("=" * 75)

VENV_PYTHON = None

for env in ENVIRONMENTS:
    candidate = env / "Scripts" / "python.exe"
    if candidate.exists():
        VENV_PYTHON = candidate
        print("Environment found:", env.name)
        break

if VENV_PYTHON is None:
    raise FileNotFoundError(
        "No deployment Python found. Checked: "
        + ", ".join(str(p) for p in ENVIRONMENTS)
    )

print("Python:", VENV_PYTHON)

packages = [
    "streamlit",
    "pandas",
    "numpy",
    "sklearn",
    "statsmodels",
    "joblib",
    "plotly"
]

check_code = """
import importlib
packages = [
    'streamlit', 'pandas', 'numpy',
    'sklearn', 'statsmodels', 'joblib', 'plotly'
]
for package in packages:
    module = importlib.import_module(package)
    print(package, getattr(module, '__version__', 'available'))
print('ALL DEPLOYMENT PACKAGES AVAILABLE')
"""

result = subprocess.run(
    [str(VENV_PYTHON), "-c", check_code],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Deployment package validation failed.")

if not TEST_SCRIPT.exists():
    raise FileNotFoundError(
        f"Validation script missing: {TEST_SCRIPT}"
    )

print("\nRunning clean environment validation script...")

test_result = subprocess.run(
    [str(VENV_PYTHON), str(TEST_SCRIPT)],
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT)
)

print(test_result.stdout)

if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("Clean environment test failed.")

print("=" * 75)
print("CLEAN DEPLOYMENT ENVIRONMENT VALIDATION PASSED")
print("=" * 75)


FINAL CLEAN DEPLOYMENT ENVIRONMENT VALIDATION
Environment found: .deployment_test_env
Python: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\.deployment_test_env\Scripts\python.exe


streamlit 1.66.1
pandas 2.2.3
numpy 1.26.4
sklearn 1.4.2
statsmodels 0.14.2
joblib 1.4.2
plotly 7.1.0
ALL DEPLOYMENT PACKAGES AVAILABLE


Running clean environment validation script...


streamlit 1.66.1
pandas 2.2.3
numpy 1.26.4
scikit-learn 1.4.2
statsmodels 0.14.2
joblib 1.4.2
plotly 7.1.0
CLEAN ENVIRONMENT DEPENDENCY CHECK PASSED

CLEAN DEPLOYMENT ENVIRONMENT VALIDATION PASSED


In [5]:

from pathlib import Path
import json
import ast
import joblib
import traceback

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

CHECK_FILES = {
    "Model Loader": PROJECT_ROOT / "models" / "model_loader.py",
    "Development Config": PROJECT_ROOT / "config" / "development.json",
    "Production Config": PROJECT_ROOT / "config" / "production.json",
    "Dashboard Integration Helper": PROJECT_ROOT / "deployment" / "dashboard_model_integration.py",
    "Dashboard": PROJECT_ROOT / "app.py",
    "Model Registry": PROJECT_ROOT / "models" / "registry" / "model_registry.json",
    "Model Version Inventory": PROJECT_ROOT / "models" / "registry" / "model_version_inventory.json",
}

print("=" * 75)
print("DEPLOYMENT AUDIT")
print("=" * 75)

file_results = {}

for name, path in CHECK_FILES.items():
    exists = path.exists()
    file_results[name] = {
        "path": str(path),
        "exists": exists
    }

    print(
        f"\n{name}:",
        "FOUND" if exists else "MISSING"
    )

    if exists and path.suffix == ".py":
        source = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        try:
            ast.parse(source)
            syntax = "VALID"
        except SyntaxError as exc:
            syntax = f"INVALID: {exc}"

        file_results[name]["syntax"] = syntax
        print("Syntax:", syntax)

        if name == "Model Loader":
            print("Functions and classes:")
            tree = ast.parse(source)

            for node in tree.body:
                if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
                    print(
                        f"  Line {node.lineno}: {node.name}"
                    )

        if name in [
            "Dashboard",
            "Dashboard Integration Helper"
        ]:
            keywords = [
                "joblib.load",
                "load_model",
                "predict(",
                "feature_names_in_",
                "n_features_in_",
                "feature_engineering",
                "rolling_mean",
                "lag_1",
                "forecast",
                "model_loader"
            ]

            matches = []

            for line_number, line in enumerate(
                source.splitlines(), 1
            ):
                if any(
                    word.lower() in line.lower()
                    for word in keywords
                ):
                    matches.append(
                        (line_number, line.strip()[:200])
                    )

            file_results[name]["source_matches"] = len(matches)

            print("Relevant source lines:", len(matches))

            for line_number, line in matches[:35]:
                print(f"  {line_number}: {line}")

    elif exists and path.suffix == ".json":
        try:
            data = json.loads(
                path.read_text(encoding="utf-8")
            )

            print("JSON: VALID")

            if isinstance(data, dict):
                print("Top-level keys:", list(data.keys())[:20])
            elif isinstance(data, list):
                print("Records:", len(data))

            file_results[name]["json_valid"] = True

        except Exception as exc:
            print("JSON ERROR:", exc)
            file_results[name]["json_valid"] = False

print("\n" + "=" * 75)
print("MODEL ARTIFACT INSPECTION")
print("=" * 75)

model_dir = PROJECT_ROOT / "models"
extensions = {".joblib", ".pkl", ".pickle"}

artifacts = sorted(
    p for p in model_dir.rglob("*")
    if p.is_file()
    and p.suffix.lower() in extensions
    and not any(
        part.startswith(".") and part.endswith("env")
        for part in p.parts
    )
)

print("Model artifacts found:", len(artifacts))

model_results = []

for index, artifact in enumerate(artifacts, 1):
    record = {
        "file": str(artifact.relative_to(PROJECT_ROOT)),
        "status": "NOT_LOADED"
    }

    try:
        obj = joblib.load(artifact)

        record["status"] = "LOADED"
        record["class"] = type(obj).__name__

        features = getattr(obj, "feature_names_in_", None)
        count = getattr(obj, "n_features_in_", None)

        record["feature_count"] = (
            int(count) if count is not None else None
        )

        record["features"] = (
            [str(x) for x in features]
            if features is not None else None
        )

        print(
            f"{index:02d}. {artifact.name}: LOADED | "
            f"Features: {record['feature_count']}"
        )

    except Exception as exc:
        record["status"] = "ERROR"
        record["error"] = str(exc)
        print(f"{index:02d}. {artifact.name}: ERROR | {exc}")

    model_results.append(record)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deployment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report = {
    "task_range": "16.44-16.46",
    "files": file_results,
    "artifact_count": len(artifacts),
    "models": model_results,
    "dashboard_modified": False,
    "integration_approved": False
}

REPORT_PATH = OUTPUT_DIR / "deployment_integration_audit.json"

REPORT_PATH.write_text(
    json.dumps(report, indent=4, default=str),
    encoding="utf-8"
)

print("\n" + "=" * 75)
print("AUDIT SUMMARY")
print("=" * 75)

print(
    "Model loader:",
    "FOUND" if CHECK_FILES["Model Loader"].exists() else "MISSING"
)

print(
    "Development configuration:",
    "FOUND" if CHECK_FILES["Development Config"].exists() else "MISSING"
)

print(
    "Production configuration:",
    "FOUND" if CHECK_FILES["Production Config"].exists() else "MISSING"
)

print("Artifacts discovered:", len(artifacts))

print(
    "Successfully loaded:",
    sum(x["status"] == "LOADED" for x in model_results)
)

print(
    "Load errors:",
    sum(x["status"] == "ERROR" for x in model_results)
)

print("Dashboard modified: NO")
print("Integration audit report:", REPORT_PATH)

print("\nINTEGRATION STATUS: REQUIRES FEATURE COMPATIBILITY VERIFICATION")
print("=" * 75)


DEPLOYMENT AUDIT

Model Loader: FOUND
Syntax: VALID
Functions and classes:
  Line 13: discover_models
  Line 26: load_model
  Line 68: predict_with_model
  Line 74: load_project_config

Development Config: FOUND
JSON: VALID
Top-level keys: ['environment', 'debug', 'model_directory', 'data_directory', 'output_directory', 'log_directory', 'forecast_target', 'forecast_horizons', 'random_seed', 'monitoring_enabled']

Production Config: FOUND
JSON: VALID
Top-level keys: ['environment', 'debug', 'model_directory', 'data_directory', 'output_directory', 'log_directory', 'forecast_target', 'forecast_horizons', 'random_seed', 'monitoring_enabled']

Dashboard Integration Helper: FOUND
Syntax: VALID
Relevant source lines: 1
  49: return joblib.load(path)

Dashboard: FOUND
Syntax: VALID
Relevant source lines: 24
  16: page_title="Care Load and Placement Demand Forecasting",
  32: FORECAST_HORIZONS = [1, 7, 14, 30]
  34: st.title("Care Load and Placement Demand Forecasting")
  37: "Historical monito

06. random_forest_14_day.joblib: LOADED | Features: 23
07. random_forest_1_day.joblib: LOADED | Features: 23
08. random_forest_30_day.joblib: LOADED | Features: 23
09. random_forest_7_day.joblib: LOADED | Features: 23
10. random_forest_recursive.joblib: LOADED | Features: 23
11. arima_final.joblib: LOADED | Features: None
12. arima_validation.joblib: LOADED | Features: None
13. holt_linear_trend_final.joblib: LOADED | Features: None
14. holt_linear_trend_validation.joblib: LOADED | Features: None
15. holt_winters_final.joblib: LOADED | Features: None
16. holt_winters_validation.joblib: LOADED | Features: None


17. sarima_final.joblib: LOADED | Features: None
18. sarima_validation.joblib: LOADED | Features: None
19. simple_exponential_smoothing_final.joblib: LOADED | Features: None
20. simple_exponential_smoothing_validation.joblib: LOADED | Features: None

AUDIT SUMMARY
Model loader: FOUND
Development configuration: FOUND
Production configuration: FOUND
Artifacts discovered: 20
Successfully loaded: 20
Load errors: 0
Dashboard modified: NO
Integration audit report: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\deployment_integration_audit.json

INTEGRATION STATUS: REQUIRES FEATURE COMPATIBILITY VERIFICATION


In [6]:
from pathlib import Path
import json
import ast
import joblib
import hashlib
import traceback
from datetime import datetime

# PROJECT CONFIGURATION

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

MODEL_DIR = PROJECT_ROOT / "models"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deployment"

APP_PATH = PROJECT_ROOT / "app.py"
MODEL_LOADER_PATH = MODEL_DIR / "model_loader.py"
LIVE_FORECAST_PATH = DEPLOYMENT_DIR / "live_model_forecasting.py"
INTEGRATION_PATH = DEPLOYMENT_DIR / "dashboard_model_integration.py"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FEATURES = [
    "lag_1", "lag_2", "lag_3", "lag_7", "lag_14", "lag_30",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_30",
    "rolling_std_7", "rolling_std_14",
    "rolling_min_7", "rolling_max_7",
    "day_of_week", "day_of_month", "month", "quarter", "year",
    "day_of_year", "month_sin", "month_cos",
    "day_of_year_sin", "day_of_year_cos"
]

report = {
    "audit_timestamp": datetime.now().isoformat(),
    "project_root": str(PROJECT_ROOT),
    "model_audit": [],
    "source_audit": {},
    "feature_engineering_files": [],
    "compatibility_summary": {},
    "errors": []
}

# PROJECT PATH VALIDATION

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_ROOT}")

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Models directory not found: {MODEL_DIR}")

# MODEL DISCOVERY

model_files = sorted(MODEL_DIR.rglob("*.joblib"))

print("\nMODEL DISCOVERY")
print("=" * 70)
print("Project:", PROJECT_ROOT)
print("Model files discovered:", len(model_files))

# MODEL FEATURE INSPECTION

print("\nMODEL FEATURE INSPECTION")
print("=" * 70)

for model_path in model_files:
    item = {
        "model": model_path.name,
        "path": str(model_path),
        "load_status": "NOT_TESTED",
        "feature_count": None,
        "features": [],
        "missing_expected_features": [],
        "unexpected_features": [],
        "feature_order_matches": None,
        "sha256": None,
        "error": None
    }

    try:
        model = joblib.load(model_path)
        item["load_status"] = "LOADED"

        if hasattr(model, "feature_names_in_"):
            actual_features = [
                str(feature) for feature in model.feature_names_in_
            ]

            item["features"] = actual_features
            item["feature_count"] = len(actual_features)

            item["missing_expected_features"] = [
                feature for feature in EXPECTED_FEATURES
                if feature not in actual_features
            ]

            item["unexpected_features"] = [
                feature for feature in actual_features
                if feature not in EXPECTED_FEATURES
            ]

            item["feature_order_matches"] = (
                actual_features == EXPECTED_FEATURES
            )

        elif hasattr(model, "n_features_in_"):
            item["feature_count"] = int(model.n_features_in_)
            item["features"] = None
            item["feature_order_matches"] = (
                int(model.n_features_in_) == len(EXPECTED_FEATURES)
            )

        else:
            item["feature_count"] = None
            item["features"] = None

        digest = hashlib.sha256()

        with open(model_path, "rb") as file:
            for chunk in iter(lambda: file.read(1024 * 1024), b""):
                digest.update(chunk)

        item["sha256"] = digest.hexdigest()

    except Exception as exc:
        item["load_status"] = "FAILED"
        item["error"] = str(exc)
        report["errors"].append({
            "file": str(model_path),
            "error": str(exc)
        })

    report["model_audit"].append(item)

    print(
        f"{item['model']} | "
        f"Status: {item['load_status']} | "
        f"Features: {item['feature_count']} | "
        f"Order match: {item['feature_order_matches']}"
    )

# DASHBOARD SOURCE VALIDATION

print("\nDASHBOARD SOURCE VALIDATION")
print("=" * 70)

source_paths = {
    "dashboard": APP_PATH,
    "model_loader": MODEL_LOADER_PATH,
    "live_forecasting": LIVE_FORECAST_PATH,
    "dashboard_integration": INTEGRATION_PATH
}

source_contents = {}

for source_name, source_path in source_paths.items():
    result = {
        "path": str(source_path),
        "exists": source_path.exists(),
        "syntax_valid": None,
        "line_count": None,
        "relevant_lines": [],
        "error": None
    }

    if source_path.exists():
        try:
            source = source_path.read_text(
                encoding="utf-8",
                errors="replace"
            )

            source_contents[source_name] = source
            result["line_count"] = len(source.splitlines())

            if source_path.suffix == ".py":
                ast.parse(source)
                result["syntax_valid"] = True

            keywords = [
                "feature", "lag_", "rolling_", "predict",
                "joblib.load", "load_model", "forecast",
                "render_live_forecasting_panel",
                "feature_names_in_", "n_features_in_"
            ]

            for line_number, line in enumerate(
                source.splitlines(), start=1
            ):
                if any(word.lower() in line.lower() for word in keywords):
                    result["relevant_lines"].append({
                        "line": line_number,
                        "text": line.strip()[:300]
                    })

        except Exception as exc:
            result["error"] = str(exc)
            report["errors"].append({
                "file": str(source_path),
                "error": str(exc)
            })

    report["source_audit"][source_name] = result

    print(
        f"{source_name}: "
        f"{'FOUND' if result['exists'] else 'MISSING'} | "
        f"Syntax: {result['syntax_valid']}"
    )

# FEATURE ENGINEERING DISCOVERY

print("\nFEATURE ENGINEERING DISCOVERY")
print("=" * 70)

search_directories = [
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "deployment",
    PROJECT_ROOT / "notebooks"
]

feature_keywords = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "day_of_year_sin",
    "month_sin",
    "lag_1",
    "feature_names_in_"
]

for directory in search_directories:
    if not directory.exists():
        continue

    for source_path in directory.rglob("*.py"):
        try:
            source = source_path.read_text(
                encoding="utf-8",
                errors="replace"
            )

            matches = [
                keyword for keyword in feature_keywords
                if keyword in source
            ]

            if matches:
                record = {
                    "path": str(source_path),
                    "matched_features": matches
                }

                report["feature_engineering_files"].append(record)

                print(source_path.relative_to(PROJECT_ROOT))
                print("  Matched:", ", ".join(matches))

        except Exception as exc:
            report["errors"].append({
                "file": str(source_path),
                "error": str(exc)
            })

# COMPATIBILITY CHECK

print("\nCOMPATIBILITY CHECK")
print("=" * 70)

loaded_models = [
    item for item in report["model_audit"]
    if item["load_status"] == "LOADED"
]

ml_models = [
    item for item in loaded_models
    if item["features"] is not None
]

statistical_models = [
    item for item in loaded_models
    if item["features"] is None
]

compatible_models = []
incompatible_models = []

for item in ml_models:
    if (
        item["feature_count"] == len(EXPECTED_FEATURES)
        and not item["missing_expected_features"]
        and not item["unexpected_features"]
        and item["feature_order_matches"]
    ):
        compatible_models.append(item["model"])
    else:
        incompatible_models.append({
            "model": item["model"],
            "feature_count": item["feature_count"],
            "missing_features": item["missing_expected_features"],
            "unexpected_features": item["unexpected_features"],
            "feature_order_matches": item["feature_order_matches"]
        })

print("Loaded models:", len(loaded_models))
print("Machine learning models:", len(ml_models))
print("Statistical models:", len(statistical_models))
print("ML models with exact expected feature order:", len(compatible_models))
print("ML models requiring investigation:", len(incompatible_models))

# LIVE FORECASTING SOURCE REVIEW

print("\nLIVE FORECASTING SOURCE REVIEW")
print("=" * 70)

live_source = source_contents.get("live_forecasting", "")

if live_source:
    print("File:", LIVE_FORECAST_PATH)
    print("Full source follows:\n")
    print(live_source)
else:
    print("Live forecasting source is missing or unavailable.")

# MODEL LOADER SOURCE REVIEW

print("\nMODEL LOADER SOURCE REVIEW")
print("=" * 70)

loader_source = source_contents.get("model_loader", "")

if loader_source:
    print("File:", MODEL_LOADER_PATH)
    print(loader_source)
else:
    print("Model loader source is missing or unavailable.")

# REPORT GENERATION

report["compatibility_summary"] = {
    "models_discovered": len(model_files),
    "models_loaded": len(loaded_models),
    "models_failed": len(model_files) - len(loaded_models),
    "machine_learning_models": len(ml_models),
    "statistical_models": len(statistical_models),
    "exactly_compatible_ml_models": len(compatible_models),
    "ml_models_requiring_investigation": incompatible_models,
    "expected_feature_count": len(EXPECTED_FEATURES),
    "expected_features": EXPECTED_FEATURES,
    "feature_engineering_files_found": len(
        report["feature_engineering_files"]
    )
}

REPORT_PATH = (
    OUTPUT_DIR / "live_dashboard_feature_compatibility.json"
)

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(report, file, indent=4, ensure_ascii=False)

# FINAL VALIDATION SUMMARY

print("\nFINAL VALIDATION SUMMARY")
print("=" * 70)

print("Models discovered:", len(model_files))
print("Models loaded:", len(loaded_models))
print("Models failed:", len(model_files) - len(loaded_models))
print("Compatible ML models:", len(compatible_models))
print("ML models requiring investigation:", len(incompatible_models))
print("Feature engineering files:", len(report["feature_engineering_files"]))
print("Source files inspected:", len(report["source_audit"]))
print("Audit errors:", len(report["errors"]))

print("\nAudit report saved:")
print(REPORT_PATH)

if incompatible_models:
    print("\nCOMPATIBILITY INVESTIGATION REQUIRED")
    for item in incompatible_models:
        print(
            item["model"],
            "| Missing:", item["missing_features"],
            "| Unexpected:", item["unexpected_features"],
            "| Order match:", item["feature_order_matches"]
        )
else:
    print("\nAll inspected ML models match the expected feature schema.")

print("\nAUDIT COMPLETED")


MODEL DISCOVERY
Project: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand
Model files discovered: 20

MODEL FEATURE INSPECTION
gradient_boosting_14_day.joblib | Status: LOADED | Features: 23 | Order match: True
gradient_boosting_1_day.joblib | Status: LOADED | Features: 23 | Order match: True
gradient_boosting_30_day.joblib | Status: LOADED | Features: 23 | Order match: True
gradient_boosting_7_day.joblib | Status: LOADED | Features: 23 | Order match: True
gradient_boosting_recursive.joblib | Status: LOADED | Features: 23 | Order match: True
random_forest_14_day.joblib | Status: LOADED | Features: 23 | Order match: True


random_forest_1_day.joblib | Status: LOADED | Features: 23 | Order match: True
random_forest_30_day.joblib | Status: LOADED | Features: 23 | Order match: True
random_forest_7_day.joblib | Status: LOADED | Features: 23 | Order match: True
random_forest_recursive.joblib | Status: LOADED | Features: 23 | Order match: True


arima_final.joblib | Status: LOADED | Features: None | Order match: None
arima_validation.joblib | Status: LOADED | Features: None | Order match: None
holt_linear_trend_final.joblib | Status: LOADED | Features: None | Order match: None
holt_linear_trend_validation.joblib | Status: LOADED | Features: None | Order match: None
holt_winters_final.joblib | Status: LOADED | Features: None | Order match: None
holt_winters_validation.joblib | Status: LOADED | Features: None | Order match: None


sarima_final.joblib | Status: LOADED | Features: None | Order match: None
sarima_validation.joblib | Status: LOADED | Features: None | Order match: None


simple_exponential_smoothing_final.joblib | Status: LOADED | Features: None | Order match: None
simple_exponential_smoothing_validation.joblib | Status: LOADED | Features: None | Order match: None

DASHBOARD SOURCE VALIDATION
dashboard: FOUND | Syntax: True
model_loader: FOUND | Syntax: True
live_forecasting: FOUND | Syntax: True
dashboard_integration: FOUND | Syntax: True

FEATURE ENGINEERING DISCOVERY
deployment\live_model_forecasting.py
  Matched: rolling_mean_7, rolling_mean_14, rolling_mean_30, day_of_year_sin, month_sin, lag_1, feature_names_in_
deployment\live_model_forecasting_backup.py
  Matched: rolling_mean_7, rolling_mean_14, rolling_mean_30, day_of_year_sin, month_sin, lag_1, feature_names_in_
deployment\live_model_forecasting_backup_final.py
  Matched: rolling_mean_7, rolling_mean_14, rolling_mean_30, day_of_year_sin, month_sin, lag_1, feature_names_in_

COMPATIBILITY CHECK
Loaded models: 20
Machine learning models: 10
Statistical models: 10
ML models with exact expected 

In [7]:
from pathlib import Path

# PROJECT CONFIGURATION

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

OUTPUT_PATH = (
    PROJECT_ROOT / "deployment" / "live_model_forecasting.py"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project not found: {PROJECT_ROOT}")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# CORRECTED LIVE FORECASTING MODULE

source_code = r'''
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st
import joblib

# MODEL CONFIGURATION

TARGET_COLUMN = "children_in_hhs_care"
DATE_COLUMN = "date"

ML_FILES = [
    "gradient_boosting_1_day.joblib",
    "gradient_boosting_7_day.joblib",
    "gradient_boosting_14_day.joblib",
    "gradient_boosting_30_day.joblib",
    "random_forest_1_day.joblib",
    "random_forest_7_day.joblib",
    "random_forest_14_day.joblib",
    "random_forest_30_day.joblib",
]

STAT_FILES = [
    "arima_final.joblib",
    "holt_linear_trend_final.joblib",
    "holt_winters_final.joblib",
    "sarima_final.joblib",
    "simple_exponential_smoothing_final.joblib",
]

FEATURES = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "rolling_std_7",
    "rolling_std_14",
    "rolling_min_7",
    "rolling_max_7",
    "day_of_week",
    "day_of_month",
    "month",
    "quarter",
    "year",
    "day_of_year",
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos",
]

# FEATURE ENGINEERING

def _features(frame):

    required_columns = [DATE_COLUMN, TARGET_COLUMN]

    missing_columns = [
        column for column in required_columns
        if column not in frame.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Required dataset columns missing: {missing_columns}"
        )

    data = frame[required_columns].copy()

    data[DATE_COLUMN] = pd.to_datetime(
        data[DATE_COLUMN],
        errors="coerce"
    )

    data[TARGET_COLUMN] = pd.to_numeric(
        data[TARGET_COLUMN],
        errors="coerce"
    )

    data = (
        data.dropna(subset=required_columns)
        .sort_values(DATE_COLUMN)
        .drop_duplicates(subset=[DATE_COLUMN], keep="last")
        .reset_index(drop=True)
    )

    values = data[TARGET_COLUMN]

    if len(data) < 31:
        raise ValueError(
            "At least 31 valid historical observations are required."
        )

    date = data[DATE_COLUMN].iloc[-1]

    prior = values.iloc[:-1]

    row = {}

    for lag in [1, 2, 3, 7, 14, 30]:
        row[f"lag_{lag}"] = float(values.iloc[-lag])

    for window in [7, 14, 30]:

        sample = prior.tail(window)

        if len(sample) < window:
            raise ValueError(
                f"Insufficient observations for rolling window {window}."
            )

        row[f"rolling_mean_{window}"] = float(sample.mean())

    sample_7 = prior.tail(7)
    sample_14 = prior.tail(14)

    row["rolling_std_7"] = float(
        sample_7.std(ddof=1)
    )

    row["rolling_std_14"] = float(
        sample_14.std(ddof=1)
    )

    row["rolling_min_7"] = float(
        sample_7.min()
    )

    row["rolling_max_7"] = float(
        sample_7.max()
    )

    row["day_of_week"] = int(date.dayofweek)
    row["day_of_month"] = int(date.day)
    row["month"] = int(date.month)
    row["quarter"] = int(date.quarter)
    row["year"] = int(date.year)
    row["day_of_year"] = int(date.dayofyear)

    row["month_sin"] = float(
        np.sin(2 * np.pi * date.month / 12)
    )

    row["month_cos"] = float(
        np.cos(2 * np.pi * date.month / 12)
    )

    row["day_of_year_sin"] = float(
        np.sin(2 * np.pi * date.dayofyear / 365.25)
    )

    row["day_of_year_cos"] = float(
        np.cos(2 * np.pi * date.dayofyear / 365.25)
    )

    features = pd.DataFrame(
        [row],
        columns=FEATURES
    )

    if features.shape[1] != 23:
        raise ValueError(
            f"Expected 23 features, generated {features.shape[1]}."
        )

    if features.isna().any().any():
        missing = features.columns[
            features.isna().any()
        ].tolist()

        raise ValueError(
            f"Generated features contain missing values: {missing}"
        )

    if not np.isfinite(features.to_numpy(dtype=float)).all():
        raise ValueError(
            "Generated features contain non-finite values."
        )

    return features


# MODEL FEATURE VALIDATION

def _validate_model_features(model, features):

    model_features = getattr(
        model,
        "feature_names_in_",
        None
    )

    if model_features is not None:

        model_features = list(model_features)

        missing = [
            name for name in model_features
            if name not in features.columns
        ]

        if missing:
            raise ValueError(
                f"Missing model input features: {missing}"
            )

        features = features.loc[:, model_features]

    expected_count = getattr(
        model,
        "n_features_in_",
        None
    )

    if (
        expected_count is not None
        and features.shape[1] != int(expected_count)
    ):
        raise ValueError(
            f"Model expects {expected_count} features, "
            f"but received {features.shape[1]}."
        )

    return features


# LIVE FORECASTING DASHBOARD

def render_live_forecasting_panel(project_root):

    project_root = Path(project_root)

    data_path = (
        project_root
        / "data"
        / "processed"
        / "cleaned_care_data.csv"
    )

    ml_dir = project_root / "models" / "machine_learning"
    stat_dir = project_root / "models" / "statistical"

    st.divider()
    st.subheader("Live Trained-Model Forecasting")

    st.caption(
        "Forecasts are generated using saved trained models "
        "and the latest available historical observations."
    )

    try:

        if not data_path.exists():
            st.error(
                f"Dataset not found: {data_path}"
            )
            return

        frame = pd.read_csv(data_path)

        frame[DATE_COLUMN] = pd.to_datetime(
            frame[DATE_COLUMN],
            errors="coerce"
        )

        frame[TARGET_COLUMN] = pd.to_numeric(
            frame[TARGET_COLUMN],
            errors="coerce"
        )

        frame = (
            frame.dropna(
                subset=[DATE_COLUMN, TARGET_COLUMN]
            )
            .sort_values(DATE_COLUMN)
            .drop_duplicates(
                subset=[DATE_COLUMN],
                keep="last"
            )
            .reset_index(drop=True)
        )

        if len(frame) < 31:
            st.warning(
                "At least 31 historical observations are required."
            )
            return

        ml_available = [
            name for name in ML_FILES
            if (ml_dir / name).exists()
        ]

        stat_available = [
            name for name in STAT_FILES
            if (stat_dir / name).exists()
        ]

        if not ml_available and not stat_available:
            st.error(
                "No supported live forecasting models were found."
            )
            return

        model_options = [
            "Gradient Boosting",
            "Random Forest",
            "Statistical Models",
        ]

        selected_family = st.selectbox(
            "Select model family",
            model_options,
            key="live_forecast_model_family"
        )

        horizon = st.selectbox(
            "Forecast horizon",
            [1, 7, 14, 30],
            key="live_forecast_horizon"
        )

        if st.button(
            "Generate live forecast",
            key="generate_live_forecast"
        ):

            results = []

            # MACHINE LEARNING FORECAST

            if selected_family in [
                "Gradient Boosting",
                "Random Forest"
            ]:

                prefix = (
                    "gradient_boosting"
                    if selected_family == "Gradient Boosting"
                    else "random_forest"
                )

                filename = f"{prefix}_{horizon}_day.joblib"
                model_path = ml_dir / filename

                if not model_path.exists():
                    st.error(
                        f"Model artifact not found: {filename}"
                    )
                    return

                model = joblib.load(model_path)

                features = _features(frame)

                features = _validate_model_features(
                    model,
                    features
                )

                prediction = model.predict(features)

                value = float(
                    np.asarray(prediction).reshape(-1)[0]
                )

                if not np.isfinite(value):
                    raise ValueError(
                        "The ML model generated a non-finite forecast."
                    )

                results.append({
                    "Model": selected_family,
                    "Horizon (observations)": horizon,
                    "Forecast": value
                })

            # STATISTICAL MODEL FORECAST

            else:

                for filename in stat_available:

                    try:

                        model = joblib.load(
                            stat_dir / filename
                        )

                        if hasattr(model, "forecast"):

                            forecast = model.forecast(
                                steps=horizon
                            )

                        elif hasattr(model, "predict"):

                            forecast = model.predict(
                                start=len(frame),
                                end=len(frame) + horizon - 1
                            )

                        else:
                            raise TypeError(
                                "Model does not support forecast() or predict()."
                            )

                        values = np.asarray(
                            forecast
                        ).reshape(-1)

                        if len(values) < horizon:
                            raise ValueError(
                                "Model returned fewer forecast values "
                                "than the requested horizon."
                            )

                        value = float(values[horizon - 1])

                        if not np.isfinite(value):
                            raise ValueError(
                                "Model returned a non-finite forecast."
                            )

                        results.append({
                            "Model": filename.replace(
                                "_final.joblib", ""
                            ),
                            "Horizon (observations)": horizon,
                            "Forecast": value
                        })

                    except Exception as exc:

                        st.warning(
                            f"{filename}: {exc}"
                        )

            # FORECAST RESULTS

            if results:

                result_frame = pd.DataFrame(results)

                st.dataframe(
                    result_frame,
                    use_container_width=True
                )

                for _, result in result_frame.iterrows():

                    st.metric(
                        f"{result['Model']} forecast",
                        f"{result['Forecast']:,.2f}"
                    )

                st.download_button(
                    "Download live forecast",
                    data=result_frame.to_csv(
                        index=False
                    ).encode("utf-8"),
                    file_name="live_model_forecast.csv",
                    mime="text/csv"
                )

            else:

                st.error(
                    "No valid forecasts were generated."
                )

    except Exception as exc:

        st.error(
            f"Live forecasting error: {exc}"
        )

        st.exception(exc)
'''

# SOURCE VALIDATION

import ast

ast.parse(source_code)

# FEATURE SCHEMA VALIDATION

expected_features = [
    "lag_1", "lag_2", "lag_3", "lag_7", "lag_14", "lag_30",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_30",
    "rolling_std_7", "rolling_std_14",
    "rolling_min_7", "rolling_max_7",
    "day_of_week", "day_of_month", "month", "quarter", "year",
    "day_of_year", "month_sin", "month_cos",
    "day_of_year_sin", "day_of_year_cos"
]

if len(expected_features) != 23:
    raise ValueError("Expected feature schema must contain 23 features.")

if '"rolling_std_14"' not in source_code:
    raise ValueError("The missing rolling_std_14 feature was not added.")

# BACKUP AND FILE UPDATE

backup_path = OUTPUT_PATH.with_name(
    "live_model_forecasting_backup.py"
)

if OUTPUT_PATH.exists() and not backup_path.exists():
    backup_path.write_text(
        OUTPUT_PATH.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

OUTPUT_PATH.write_text(
    source_code,
    encoding="utf-8"
)

# FINAL VALIDATION

saved_source = OUTPUT_PATH.read_text(
    encoding="utf-8"
)

compile(saved_source, str(OUTPUT_PATH), "exec")

print("\nLIVE FORECASTING MODULE UPDATED")
print("=" * 65)
print("File:", OUTPUT_PATH)
print("Backup:", backup_path if backup_path.exists() else "Not required")
print("Python syntax: VALID")
print("Expected ML features:", len(expected_features))
print("Missing rolling_std_14: FIXED")
print("Model artifacts modified: NO")
print("Dashboard app.py modified: NO")
print("\nMODULE VALIDATION PASSED")


LIVE FORECASTING MODULE UPDATED
File: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\live_model_forecasting.py
Backup: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\live_model_forecasting_backup.py
Python syntax: VALID
Expected ML features: 23
Missing rolling_std_14: FIXED
Model artifacts modified: NO
Dashboard app.py modified: NO

MODULE VALIDATION PASSED


In [8]:
from pathlib import Path
from datetime import datetime
import ast
import json
import shutil
import importlib.util
import joblib
import numpy as np
import pandas as pd
import traceback

# PROJECT CONFIGURATION

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_care_data.csv"
MODULE_PATH = PROJECT_ROOT / "deployment" / "live_model_forecasting.py"
APP_PATH = PROJECT_ROOT / "app.py"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deployment"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# MODEL CONFIGURATION

ML_MODELS = [
    "gradient_boosting_1_day.joblib",
    "gradient_boosting_7_day.joblib",
    "gradient_boosting_14_day.joblib",
    "gradient_boosting_30_day.joblib",
    "random_forest_1_day.joblib",
    "random_forest_7_day.joblib",
    "random_forest_14_day.joblib",
    "random_forest_30_day.joblib",
]

STAT_MODELS = [
    "arima_final.joblib",
    "holt_linear_trend_final.joblib",
    "holt_winters_final.joblib",
    "sarima_final.joblib",
    "simple_exponential_smoothing_final.joblib",
]

EXPECTED_FEATURES = [
    "lag_1", "lag_2", "lag_3", "lag_7", "lag_14", "lag_30",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_30",
    "rolling_std_7", "rolling_std_14",
    "rolling_min_7", "rolling_max_7",
    "day_of_week", "day_of_month", "month", "quarter", "year",
    "day_of_year", "month_sin", "month_cos",
    "day_of_year_sin", "day_of_year_cos"
]

# PROJECT VALIDATION

for path in [PROJECT_ROOT, DATA_PATH, MODULE_PATH, APP_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

# MODULE BACKUP AND SOURCE VALIDATION

backup_path = MODULE_PATH.with_name(
    "live_model_forecasting_backup_final.py"
)

if not backup_path.exists():
    shutil.copy2(MODULE_PATH, backup_path)

module_source = MODULE_PATH.read_text(encoding="utf-8")

ast.parse(module_source)

if "rolling_std_14" not in module_source:
    raise ValueError(
        "Corrected rolling_std_14 feature is missing from live module."
    )

print("\nMODULE VALIDATION")
print("=" * 65)
print("Live module syntax: VALID")
print("23-feature schema correction: PRESENT")
print("Backup:", backup_path)

# DATASET VALIDATION

frame = pd.read_csv(DATA_PATH)

required_columns = ["date", "children_in_hhs_care"]

missing_columns = [
    column for column in required_columns
    if column not in frame.columns
]

if missing_columns:
    raise ValueError(f"Dataset columns missing: {missing_columns}")

frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
frame["children_in_hhs_care"] = pd.to_numeric(
    frame["children_in_hhs_care"],
    errors="coerce"
)

frame = (
    frame.dropna(subset=required_columns)
    .sort_values("date")
    .drop_duplicates(subset=["date"], keep="last")
    .reset_index(drop=True)
)

if len(frame) < 31:
    raise ValueError("At least 31 valid observations are required.")

print("\nDATASET VALIDATION")
print("=" * 65)
print("Observations:", len(frame))
print("Date range:", frame["date"].min(), "to", frame["date"].max())

# LIVE MODULE IMPORT

spec = importlib.util.spec_from_file_location(
    "live_model_forecasting_final",
    MODULE_PATH
)

live_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(live_module)

# FEATURE GENERATION AND VALIDATION

features = live_module._features(frame)

if list(features.columns) != EXPECTED_FEATURES:
    raise ValueError("Generated feature names or order do not match.")

if features.shape != (1, 23):
    raise ValueError(f"Unexpected feature shape: {features.shape}")

if features.isna().any().any():
    raise ValueError("Generated features contain missing values.")

if not np.isfinite(features.to_numpy(dtype=float)).all():
    raise ValueError("Generated features contain non-finite values.")

print("\nFEATURE VALIDATION")
print("=" * 65)
print("Generated features:", features.shape[1])
print("Feature order: VALID")
print("Missing values:", int(features.isna().sum().sum()))
print("rolling_std_14:", features["rolling_std_14"].iloc[0])

# MODEL DISCOVERY

def find_model_file(filename):
    matches = [
        path for path in MODEL_DIR.rglob(filename)
        if path.is_file() and "registry" not in path.parts
    ]

    if len(matches) > 1:
        raise RuntimeError(
            f"Duplicate model artifact found: {filename}: {matches}"
        )

    return matches[0] if matches else None


ml_paths = {
    filename: find_model_file(filename)
    for filename in ML_MODELS
}

stat_paths = {
    filename: find_model_file(filename)
    for filename in STAT_MODELS
}

print("\nMODEL DISCOVERY")
print("=" * 65)
print("ML artifacts found:", sum(p is not None for p in ml_paths.values()), "/", len(ML_MODELS))
print("Statistical artifacts found:", sum(p is not None for p in stat_paths.values()), "/", len(STAT_MODELS))

# MACHINE LEARNING FORECAST TEST

ml_results = []
ml_errors = []

print("\nMACHINE LEARNING FORECAST TEST")
print("=" * 65)

for filename, model_path in ml_paths.items():

    if model_path is None:
        ml_errors.append({
            "model": filename,
            "error": "Artifact not found"
        })
        print(filename, "| MISSING")
        continue

    try:
        model = joblib.load(model_path)

        model_input = live_module._validate_model_features(
            model,
            features.copy()
        )

        prediction = model.predict(model_input)

        forecast = float(
            np.asarray(prediction).reshape(-1)[0]
        )

        if not np.isfinite(forecast):
            raise ValueError("Non-finite forecast.")

        ml_results.append({
            "Model": filename,
            "Features": model_input.shape[1],
            "Forecast": forecast,
            "Status": "PASSED"
        })

        print(
            filename,
            "| Features:", model_input.shape[1],
            "| Forecast:", round(forecast, 4),
            "| PASSED"
        )

    except Exception as exc:
        ml_errors.append({
            "model": filename,
            "error": str(exc)
        })
        print(filename, "| FAILED:", exc)

# STATISTICAL FORECAST TEST

stat_results = []
stat_errors = []

print("\nSTATISTICAL FORECAST TEST")
print("=" * 65)

for filename, model_path in stat_paths.items():

    if model_path is None:
        stat_errors.append({
            "model": filename,
            "error": "Artifact not found"
        })
        print(filename, "| MISSING")
        continue

    try:
        model = joblib.load(model_path)

        if hasattr(model, "forecast"):
            forecast_values = model.forecast(steps=30)

        elif hasattr(model, "predict"):
            forecast_values = model.predict(
                start=len(frame),
                end=len(frame) + 29
            )

        else:
            raise TypeError("Unsupported statistical model.")

        values = np.asarray(forecast_values).reshape(-1)

        if len(values) < 30:
            raise ValueError("Fewer than 30 forecast values returned.")

        if not np.isfinite(values[:30]).all():
            raise ValueError("Non-finite forecast values returned.")

        stat_results.append({
            "Model": filename,
            "Forecast_1": float(values[0]),
            "Forecast_7": float(values[6]),
            "Forecast_14": float(values[13]),
            "Forecast_30": float(values[29]),
            "Status": "PASSED"
        })

        print(filename, "| 30-step forecast: PASSED")

    except Exception as exc:
        stat_errors.append({
            "model": filename,
            "error": str(exc)
        })
        print(filename, "| FAILED:", exc)

# DASHBOARD INTEGRATION VALIDATION

print("\nDASHBOARD INTEGRATION VALIDATION")
print("=" * 65)

app_source = APP_PATH.read_text(
    encoding="utf-8",
    errors="replace"
)

ast.parse(app_source)

dashboard_import_present = (
    "render_live_forecasting_panel" in app_source
)

expected_ml_dir = MODEL_DIR / "machine_learning"
expected_stat_dir = MODEL_DIR / "statistical"

ml_path_issues = [
    {
        "model": filename,
        "actual_path": str(path)
    }
    for filename, path in ml_paths.items()
    if path is not None and path.parent != expected_ml_dir
]

stat_path_issues = [
    {
        "model": filename,
        "actual_path": str(path)
    }
    for filename, path in stat_paths.items()
    if path is not None and path.parent != expected_stat_dir
]

print("Dashboard syntax: VALID")
print("Live forecasting integration reference:", dashboard_import_present)
print("ML artifact directory:", expected_ml_dir)
print("Statistical artifact directory:", expected_stat_dir)
print("ML path mismatches:", len(ml_path_issues))
print("Statistical path mismatches:", len(stat_path_issues))

# DEPLOYMENT REPORT GENERATION

all_ml_passed = (
    len(ml_results) == len(ML_MODELS)
    and len(ml_errors) == 0
)

all_stat_passed = (
    len(stat_results) == len(STAT_MODELS)
    and len(stat_errors) == 0
)

all_paths_valid = (
    len(ml_path_issues) == 0
    and len(stat_path_issues) == 0
)

overall_passed = (
    all_ml_passed
    and all_stat_passed
    and all_paths_valid
    and dashboard_import_present
)

report = {
    "timestamp": datetime.now().isoformat(),
    "project": str(PROJECT_ROOT),
    "dataset": {
        "path": str(DATA_PATH),
        "observations": len(frame),
        "start_date": str(frame["date"].min()),
        "end_date": str(frame["date"].max())
    },
    "module": {
        "syntax_valid": True,
        "feature_count": len(features.columns),
        "backup": str(backup_path)
    },
    "feature_validation": {
        "passed": True,
        "features": EXPECTED_FEATURES
    },
    "machine_learning": {
        "passed": len(ml_results),
        "failed": len(ml_errors),
        "results": ml_results,
        "errors": ml_errors
    },
    "statistical": {
        "passed": len(stat_results),
        "failed": len(stat_errors),
        "results": stat_results,
        "errors": stat_errors
    },
    "dashboard": {
        "syntax_valid": True,
        "live_integration_reference": dashboard_import_present,
        "ml_path_mismatches": ml_path_issues,
        "statistical_path_mismatches": stat_path_issues
    },
    "overall_status": (
        "PASSED" if overall_passed else "REVIEW REQUIRED"
    )
}

REPORT_PATH = OUTPUT_DIR / "final_live_deployment_validation.json"

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(report, file, indent=4, ensure_ascii=False)

# FINAL VALIDATION SUMMARY

print("\nFINAL VALIDATION SUMMARY")
print("=" * 65)
print("Feature generation: PASSED")
print(f"ML forecasts: {len(ml_results)}/{len(ML_MODELS)}")
print(f"Statistical forecasts: {len(stat_results)}/{len(STAT_MODELS)}")
print("Dashboard integration reference:", "PASSED" if dashboard_import_present else "REVIEW")
print("Dashboard artifact paths:", "PASSED" if all_paths_valid else "REVIEW")
print("Report:", REPORT_PATH)

if overall_passed:
    print("\nLIVE DEPLOYMENT VALIDATION PASSED")
else:
    print("\nLIVE DEPLOYMENT VALIDATION NEEDS REVIEW")

    if ml_errors:
        print("\nML ERRORS:", ml_errors)

    if stat_errors:
        print("\nSTATISTICAL ERRORS:", stat_errors)

    if ml_path_issues:
        print("\nML PATH ISSUES:", ml_path_issues)

    if stat_path_issues:
        print("\nSTATISTICAL PATH ISSUES:", stat_path_issues)


MODULE VALIDATION
Live module syntax: VALID
23-feature schema correction: PRESENT
Backup: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\deployment\live_model_forecasting_backup_final.py

DATASET VALIDATION
Observations: 720
Date range: 2023-01-12 00:00:00 to 2025-12-21 00:00:00



FEATURE VALIDATION
Generated features: 23
Feature order: VALID
Missing values: 0
rolling_std_14: 24.55684144059373

MODEL DISCOVERY
ML artifacts found: 8 / 8
Statistical artifacts found: 5 / 5

MACHINE LEARNING FORECAST TEST
gradient_boosting_1_day.joblib | Features: 23 | Forecast: 5877.1438 | PASSED
gradient_boosting_7_day.joblib | Features: 23 | Forecast: 6334.7256 | PASSED
gradient_boosting_14_day.joblib | Features: 23 | Forecast: 6570.137 | PASSED
gradient_boosting_30_day.joblib | Features: 23 | Forecast: 6432.2752 | PASSED
random_forest_1_day.joblib | Features: 23 | Forecast: 5927.4878 | PASSED
random_forest_7_day.joblib | Features: 23 | Forecast: 6495.6 | PASSED


random_forest_14_day.joblib | Features: 23 | Forecast: 6665.596 | PASSED
random_forest_30_day.joblib | Features: 23 | Forecast: 6534.9432 | PASSED

STATISTICAL FORECAST TEST
arima_final.joblib | 30-step forecast: PASSED
holt_linear_trend_final.joblib | 30-step forecast: PASSED
holt_winters_final.joblib | 30-step forecast: PASSED
sarima_final.joblib | 30-step forecast: PASSED
simple_exponential_smoothing_final.joblib | 30-step forecast: PASSED

DASHBOARD INTEGRATION VALIDATION
Dashboard syntax: VALID
Live forecasting integration reference: False
ML artifact directory: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\models\machine_learning
Statistical artifact directory: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\models\statistical
ML path mismatches: 0
Statistical path mismatches: 0

FINAL VALIDATION SUMMARY
Feature generation: PASSED
ML forecasts: 8/8
Statistical forecasts: 5/5
Dashboard integration reference: REVIEW
Dashboard artifact paths: 

In [9]:
from pathlib import Path
import subprocess
import socket
import time
import webbrowser
import os

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

APP_PATH = PROJECT_ROOT / "app.py"

DEPLOYMENT_PYTHON = (
    PROJECT_ROOT / ".deployment_test_env" / "Scripts" / "python.exe"
)

LOG_DIR = PROJECT_ROOT / "outputs" / "deployment"
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = LOG_DIR / "streamlit_live_dashboard.log"

HOST = "127.0.0.1"
PORT = 8501
DASHBOARD_URL = f"http://{HOST}:{PORT}"

if not APP_PATH.exists():
    raise FileNotFoundError(f"Dashboard application not found: {APP_PATH}")

if not DEPLOYMENT_PYTHON.exists():
    raise FileNotFoundError(
        f"Deployment Python not found: {DEPLOYMENT_PYTHON}"
    )

def is_port_open(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0

if is_port_open(HOST, PORT):
    print("Streamlit dashboard is already running.")
    print("Dashboard URL:", DASHBOARD_URL)
    webbrowser.open(DASHBOARD_URL)

else:
    log_file = open(LOG_PATH, "a", encoding="utf-8")

    process = subprocess.Popen(
        [
            str(DEPLOYMENT_PYTHON),
            "-m",
            "streamlit",
            "run",
            str(APP_PATH),
            "--server.address",
            HOST,
            "--server.port",
            str(PORT),
            "--server.headless",
            "true"
        ],
        cwd=str(PROJECT_ROOT),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NEW_PROCESS_GROUP
    )

    for attempt in range(20):
        time.sleep(1)

        if is_port_open(HOST, PORT):
            break

        if process.poll() is not None:
            log_file.close()
            raise RuntimeError(
                f"Streamlit stopped unexpectedly. Check log: {LOG_PATH}"
            )

    log_file.close()

    if is_port_open(HOST, PORT):
        print("STREAMLIT DASHBOARD STARTED SUCCESSFULLY")
        print("Dashboard URL:", DASHBOARD_URL)
        print("Application:", APP_PATH)
        print("Python:", DEPLOYMENT_PYTHON)
        print("Log file:", LOG_PATH)

        webbrowser.open(DASHBOARD_URL)
    else:
        print("Dashboard did not start within the expected time.")
        print("Check log file:", LOG_PATH)

Streamlit dashboard is already running.
Dashboard URL: http://127.0.0.1:8501


In [10]:

from pathlib import Path
import subprocess
import sys
import importlib.metadata as metadata

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

VENV_PYTHON = PROJECT_ROOT / ".deployment_test_env" / "Scripts" / "python.exe"

if not VENV_PYTHON.exists():
    raise FileNotFoundError(
        f"Deployment Python not found: {VENV_PYTHON}"
    )

print("Checking deployment environment...")

subprocess.run(
    [
        str(VENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--force-reinstall",
        "--no-cache-dir",
        "pyarrow==19.0.1"
    ],
    check=True
)

print("\nValidating installed dependencies...")

validation_code = r'''
import numpy
import pandas
import pyarrow
import streamlit
import sklearn
import statsmodels

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Streamlit:", streamlit.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Statsmodels:", statsmodels.__version__)

assert numpy.__version__ == "1.26.4"
assert pyarrow.__version__ == "19.0.1"

import pandas as pd
import streamlit as st

df = pd.DataFrame({
    "Date": pd.date_range("2025-01-01", periods=5),
    "Forecast": [100, 110, 120, 130, 140]
})

print("\nDataFrame compatibility test PASSED")
print(df.to_string())

print("\nDEPLOYMENT DEPENDENCY VALIDATION PASSED")
'''

subprocess.run(
    [str(VENV_PYTHON), "-c", validation_code],
    check=True
)

print("\nPYARROW COMPATIBILITY FIX COMPLETED")


Checking deployment environment...


KeyboardInterrupt: 

In [64]:

from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

VENV_PYTHON = (
    PROJECT_ROOT
    / ".deployment_test_env"
    / "Scripts"
    / "python.exe"
)

if not VENV_PYTHON.exists():
    raise FileNotFoundError(
        f"Deployment Python not found: {VENV_PYTHON}"
    )

def run_command(command):
    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError("Command failed. Check output above.")

print("Checking active deployment interpreter...")
run_command([
    str(VENV_PYTHON), "-c",
    "import sys; print(sys.executable)"
])

print("\nChecking current package versions...")
run_command([
    str(VENV_PYTHON), "-m", "pip",
    "show", "numpy", "pyarrow"
])

print("\nRemoving incompatible PyArrow installation...")
run_command([
    str(VENV_PYTHON), "-m", "pip",
    "uninstall", "-y", "pyarrow"
])

print("\nInstalling compatible pinned versions...")
run_command([
    str(VENV_PYTHON), "-m", "pip",
    "install",
    "--no-cache-dir",
    "--force-reinstall",
    "--no-deps",
    "numpy==1.26.4",
    "pyarrow==18.1.0"
])

print("\nValidating NumPy and PyArrow...")
validation_code = r'''
import sys
import numpy as np
import pandas as pd
import pyarrow as pa
import streamlit as st

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("Streamlit:", st.__version__)

assert np.__version__ == "1.26.4"
assert pa.__version__ == "18.1.0"

df = pd.DataFrame({
    "Date": pd.date_range("2025-01-01", periods=5),
    "Forecast": [100, 110, 120, 130, 140]
})

table = pa.Table.from_pandas(df)
assert table.num_rows == 5

print("PyArrow DataFrame conversion: PASSED")
print("NUMPY-PYARROW COMPATIBILITY: PASSED")
'''

run_command([
    str(VENV_PYTHON), "-c", validation_code
])

print("\nCOMPATIBILITY VALIDATION COMPLETED.")
print("Restart Streamlit using this exact deployment interpreter.")


Checking active deployment interpreter...
C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\.deployment_test_env\Scripts\python.exe


Checking current package versions...
Name: numpy
Version: 1.26.4
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2023, NumPy Developers.
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are
met:

    * Redistributions of source code must retain the above copyright
       notice, this list of conditions and the following disclaimer.

    * Redistributions in binary form must reproduce the above
       copyright notice, this list of conditions and the following
       disclaimer in the documentation and/or other materials provided
       with the distribution.

    * Neither the name of the NumPy Developers nor the name

In [74]:
from pathlib import Path
import subprocess
import time
import socket
import webbrowser
import sys

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

PYTHON = PROJECT_ROOT / ".deployment_test_env" / "Scripts" / "python.exe"
APP_PATH = PROJECT_ROOT / "app.py"

if not PYTHON.exists():
    raise FileNotFoundError(f"Deployment Python not found: {PYTHON}")

if not APP_PATH.exists():
    raise FileNotFoundError(f"Dashboard not found: {APP_PATH}")

print("Removing existing PyArrow installation...")

subprocess.run(
    [str(PYTHON), "-m", "pip", "uninstall", "-y", "pyarrow"],
    check=False
)

site_packages = PYTHON.parent.parent / "Lib" / "site-packages"

for item in site_packages.glob("pyarrow*"):
    if item.is_dir():
        import shutil
        shutil.rmtree(item, ignore_errors=True)
    elif item.is_file():
        item.unlink(missing_ok=True)

print("Installing compatible PyArrow version...")

subprocess.run(
    [
        str(PYTHON), "-m", "pip", "install",
        "--no-cache-dir",
        "--no-deps",
        "--force-reinstall",
        "pyarrow==18.1.0"
    ],
    check=True
)

print("\nChecking NumPy and PyArrow compatibility...")

test_code = """
import numpy as np
import pandas as pd
import pyarrow as pa

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)

df = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=3),
    "value": [10, 20, 30]
})

table = pa.Table.from_pandas(df)
print("PyArrow DataFrame conversion: PASSED")
"""

test = subprocess.run(
    [str(PYTHON), "-c", test_code],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True
)

print(test.stdout)

if test.returncode != 0:
    print(test.stderr)
    raise RuntimeError(
        "PyArrow compatibility check failed. Dashboard was not launched."
    )

print("NUMPY-PYARROW COMPATIBILITY: PASSED")

HOST = "127.0.0.1"
PORT = 8502
URL = f"http://{HOST}:{PORT}"

def port_is_open(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0

if port_is_open(HOST, PORT):
    raise RuntimeError(
        f"Port {PORT} is already in use. Stop its existing Streamlit process and rerun."
    )

LOG_DIR = PROJECT_ROOT / "outputs" / "deployment"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / "streamlit_live_dashboard.log"

log_file = open(LOG_PATH, "a", encoding="utf-8")

process = subprocess.Popen(
    [
        str(PYTHON),
        "-m", "streamlit", "run",
        str(APP_PATH),
        "--server.address", HOST,
        "--server.port", str(PORT),
        "--server.headless", "true"
    ],
    cwd=str(PROJECT_ROOT),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    creationflags=subprocess.CREATE_NEW_PROCESS_GROUP
)

for attempt in range(30):
    time.sleep(1)

    if port_is_open(HOST, PORT):
        break

    if process.poll() is not None:
        log_file.close()
        raise RuntimeError(
            f"Streamlit stopped unexpectedly. Check: {LOG_PATH}"
        )

log_file.close()

if port_is_open(HOST, PORT):
    print("\nSTREAMLIT DASHBOARD STARTED")
    print("Dashboard URL:", URL)
    print("Log file:", LOG_PATH)
    webbrowser.open(URL)
else:
    raise RuntimeError(
        f"Dashboard did not start. Check log file: {LOG_PATH}"
    )

Removing existing PyArrow installation...
Installing compatible PyArrow version...

Checking NumPy and PyArrow compatibility...
NumPy: 1.26.4
Pandas: 2.2.3
PyArrow: 18.1.0
PyArrow DataFrame conversion: PASSED

NUMPY-PYARROW COMPATIBILITY: PASSED


RuntimeError: Port 8502 is already in use. Stop its existing Streamlit process and rerun.

In [76]:
import urllib.request
import webbrowser

DASHBOARD_URL = "http://127.0.0.1:8502"

try:
    response = urllib.request.urlopen(
        DASHBOARD_URL,
        timeout=5
    )

    print("Dashboard HTTP Status:", response.status)

    if response.status == 200:
        print("LIVE STREAMLIT DASHBOARD IS RUNNING")
        print("Opening dashboard:", DASHBOARD_URL)
        webbrowser.open(DASHBOARD_URL)
    else:
        print("Port is responding, but dashboard needs checking.")

except Exception as error:
    print("Dashboard is not responding on port 8502.")
    print("Details:", error)
    print("Check outputs/deployment/streamlit_live_dashboard.log")

Dashboard HTTP Status: 200
LIVE STREAMLIT DASHBOARD IS RUNNING
Opening dashboard: http://127.0.0.1:8502


In [78]:
from pathlib import Path
from urllib.request import urlopen
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
import pandas as pd
import time

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

DASHBOARD_URL = "http://127.0.0.1:8502"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deployment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def test_dashboard_response(request_id):
    start_time = time.perf_counter()

    try:
        with urlopen(DASHBOARD_URL, timeout=30) as response:
            response.read()
            response_time = (
                time.perf_counter() - start_time
            ) * 1000

            return {
                "Request_ID": request_id,
                "Status": response.status,
                "Response_Time_ms": round(response_time, 2),
                "Result": "PASSED"
            }

    except Exception as error:
        response_time = (
            time.perf_counter() - start_time
        ) * 1000

        return {
            "Request_ID": request_id,
            "Status": None,
            "Response_Time_ms": round(response_time, 2),
            "Result": f"FAILED: {str(error)}"
        }

print("Dashboard Performance Testing Started")

sequential_results = [
    test_dashboard_response(i)
    for i in range(1, 21)
]

print("Sequential request testing completed")

with ThreadPoolExecutor(max_workers=5) as executor:
    concurrent_results = list(
        executor.map(
            test_dashboard_response,
            range(21, 31)
        )
    )

print("Concurrent request testing completed")

results = sequential_results + concurrent_results

performance_df = pd.DataFrame(results)

successful = performance_df[
    performance_df["Result"] == "PASSED"
]

total_requests = len(performance_df)
successful_requests = len(successful)
failed_requests = total_requests - successful_requests

if successful_requests:
    average_response = successful["Response_Time_ms"].mean()
    minimum_response = successful["Response_Time_ms"].min()
    maximum_response = successful["Response_Time_ms"].max()
    median_response = successful["Response_Time_ms"].median()
else:
    average_response = minimum_response = None
    maximum_response = median_response = None

summary = pd.DataFrame([{
    "Test_Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Dashboard_URL": DASHBOARD_URL,
    "Total_Requests": total_requests,
    "Successful_Requests": successful_requests,
    "Failed_Requests": failed_requests,
    "Success_Rate_Percent": round(
        successful_requests / total_requests * 100, 2
    ),
    "Average_Response_ms": round(average_response, 2) if average_response is not None else None,
    "Median_Response_ms": round(median_response, 2) if median_response is not None else None,
    "Minimum_Response_ms": round(minimum_response, 2) if minimum_response is not None else None,
    "Maximum_Response_ms": round(maximum_response, 2) if maximum_response is not None else None
}])

results_path = OUTPUT_DIR / "dashboard_performance_results.csv"
summary_path = OUTPUT_DIR / "dashboard_performance_summary.csv"

performance_df.to_csv(results_path, index=False)
summary.to_csv(summary_path, index=False)

print("\nDASHBOARD PERFORMANCE SUMMARY")
display(summary)

print("\nINDIVIDUAL REQUEST RESULTS")
display(performance_df)

print("\nPerformance results saved:", results_path)
print("Performance summary saved:", summary_path)

if failed_requests == 0:
    print("\nDASHBOARD HTTP PERFORMANCE TEST: PASSED")
else:
    print("\nDASHBOARD HTTP PERFORMANCE TEST: FAILED")

print("\nPerformance testing completed.")

Dashboard Performance Testing Started
Sequential request testing completed
Concurrent request testing completed

DASHBOARD PERFORMANCE SUMMARY


,Test_Date,Dashboard_URL,Total_Requests,Successful_Requests,Failed_Requests,Success_Rate_Percent,Average_Response_ms,Median_Response_ms,Minimum_Response_ms,Maximum_Response_ms
0,2026-11-05 04:15:19,http://127.0.0.1:8502,30,30,0,100.0,15.99,17.43,4.62,31.94



INDIVIDUAL REQUEST RESULTS


,Request_ID,Status,Response_Time_ms,Result
0,1,200,18.39,PASSED
1,2,200,7.14,PASSED
2,3,200,9.51,PASSED
3,4,200,20.58,PASSED
4,5,200,25.62,PASSED
5,6,200,24.66,PASSED
6,7,200,17.28,PASSED
7,8,200,28.69,PASSED
8,9,200,18.70,PASSED
9,10,200,7.80,PASSED



Performance results saved: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\dashboard_performance_results.csv
Performance summary saved: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\dashboard_performance_summary.csv

DASHBOARD HTTP PERFORMANCE TEST: PASSED

Performance testing completed.


In [80]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import json

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

RAW_PATH = (
    PROJECT_ROOT / "data" / "raw" /
    "HHS_Unaccompanied_Alien_Children_Program.csv"
)

PROCESSED_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "cleaned_care_data.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deployment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_COLUMN = "date"

TARGET_COLUMN = "children_in_hhs_care"

REQUIRED_COLUMNS = [
    "date",
    "children_apprehended_and_placed_in_cbp_custody",
    "children_in_cbp_custody",
    "children_transferred_out_of_cbp_custody",
    "children_in_hhs_care",
    "children_discharged_from_hhs_care"
]

print("DATA REFRESH VERIFICATION")
print("=" * 65)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found: {RAW_PATH}")

if not PROCESSED_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found: {PROCESSED_PATH}"
    )

raw_df = pd.read_csv(RAW_PATH)
processed_df = pd.read_csv(PROCESSED_PATH)

raw_df.columns = raw_df.columns.str.strip()
processed_df.columns = processed_df.columns.str.strip()

print("\nDATASET INFORMATION")
print("-" * 65)

print("Raw dataset shape:", raw_df.shape)
print("Processed dataset shape:", processed_df.shape)

print("Raw dataset modified:", datetime.fromtimestamp(
    RAW_PATH.stat().st_mtime
))

print("Processed dataset modified:", datetime.fromtimestamp(
    PROCESSED_PATH.stat().st_mtime
))

raw_missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in raw_df.columns
]

processed_missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in processed_df.columns
]

if raw_missing_columns or processed_missing_columns:
    raise ValueError(
        f"Missing columns. Raw: {raw_missing_columns}; "
        f"Processed: {processed_missing_columns}"
    )

raw_df[DATE_COLUMN] = pd.to_datetime(
    raw_df[DATE_COLUMN], errors="coerce"
)

processed_df[DATE_COLUMN] = pd.to_datetime(
    processed_df[DATE_COLUMN], errors="coerce"
)

raw_valid_dates = raw_df.dropna(subset=[DATE_COLUMN])
processed_valid_dates = processed_df.dropna(
    subset=[DATE_COLUMN]
)

raw_latest_date = raw_valid_dates[DATE_COLUMN].max()
processed_latest_date = processed_valid_dates[DATE_COLUMN].max()

raw_earliest_date = raw_valid_dates[DATE_COLUMN].min()
processed_earliest_date = processed_valid_dates[DATE_COLUMN].min()

raw_duplicate_dates = raw_df[DATE_COLUMN].duplicated().sum()
processed_duplicate_dates = (
    processed_df[DATE_COLUMN].duplicated().sum()
)

raw_null_dates = raw_df[DATE_COLUMN].isna().sum()
processed_null_dates = processed_df[DATE_COLUMN].isna().sum()

raw_target_nulls = raw_df[TARGET_COLUMN].isna().sum()
processed_target_nulls = processed_df[TARGET_COLUMN].isna().sum()

raw_unique_dates = raw_df[DATE_COLUMN].nunique()
processed_unique_dates = processed_df[DATE_COLUMN].nunique()

raw_date_gaps = (
    pd.date_range(
        raw_earliest_date,
        raw_latest_date,
        freq="D"
    ).difference(
        pd.DatetimeIndex(raw_valid_dates[DATE_COLUMN].unique())
    )
    if pd.notna(raw_earliest_date) and pd.notna(raw_latest_date)
    else []
)

processed_date_gaps = (
    pd.date_range(
        processed_earliest_date,
        processed_latest_date,
        freq="D"
    ).difference(
        pd.DatetimeIndex(processed_valid_dates[DATE_COLUMN].unique())
    )
    if pd.notna(processed_earliest_date)
    and pd.notna(processed_latest_date)
    else []
)

raw_latest_target = (
    raw_valid_dates.loc[
        raw_valid_dates[DATE_COLUMN] == raw_latest_date,
        TARGET_COLUMN
    ].iloc[-1]
    if pd.notna(raw_latest_date)
    else None
)

processed_latest_target = (
    processed_valid_dates.loc[
        processed_valid_dates[DATE_COLUMN] == processed_latest_date,
        TARGET_COLUMN
    ].iloc[-1]
    if pd.notna(processed_latest_date)
    else None
)

checks = {
    "Required columns available": (
        not raw_missing_columns and not processed_missing_columns
    ),
    "Raw date values valid": raw_null_dates == 0,
    "Processed date values valid": processed_null_dates == 0,
    "Raw dataset has no duplicate dates": raw_duplicate_dates == 0,
    "Processed dataset has no duplicate dates": (
        processed_duplicate_dates == 0
    ),
    "Raw target column has no null values": raw_target_nulls == 0,
    "Processed target column has no null values": (
        processed_target_nulls == 0
    ),
    "Processed dataset latest date matches raw": (
        pd.notna(raw_latest_date)
        and pd.notna(processed_latest_date)
        and raw_latest_date == processed_latest_date
    ),
    "Processed dataset begins within raw date coverage": (
        pd.notna(raw_earliest_date)
        and pd.notna(processed_earliest_date)
        and processed_earliest_date >= raw_earliest_date
    )
}

print("\nDATE COVERAGE")
print("-" * 65)

print("Raw date range:", raw_earliest_date, "to", raw_latest_date)
print(
    "Processed date range:",
    processed_earliest_date,
    "to",
    processed_latest_date
)

print("Raw unique dates:", raw_unique_dates)
print("Processed unique dates:", processed_unique_dates)

print("Raw missing calendar dates:", len(raw_date_gaps))
print("Processed missing calendar dates:", len(processed_date_gaps))

print("\nDATA QUALITY CHECKS")
print("-" * 65)

print("Raw duplicate dates:", raw_duplicate_dates)
print("Processed duplicate dates:", processed_duplicate_dates)
print("Raw invalid dates:", raw_null_dates)
print("Processed invalid dates:", processed_null_dates)
print("Raw target null values:", raw_target_nulls)
print("Processed target null values:", processed_target_nulls)

print("\nVERIFICATION RESULTS")
print("-" * 65)

for check_name, passed in checks.items():
    print(
        f"{'PASSED' if passed else 'FAILED'}: {check_name}"
    )

all_passed = all(checks.values())

summary = {
    "verification_timestamp": datetime.now().isoformat(),
    "raw_dataset": str(RAW_PATH),
    "processed_dataset": str(PROCESSED_PATH),
    "raw_observations": int(len(raw_df)),
    "processed_observations": int(len(processed_df)),
    "raw_latest_date": str(raw_latest_date),
    "processed_latest_date": str(processed_latest_date),
    "raw_unique_dates": int(raw_unique_dates),
    "processed_unique_dates": int(processed_unique_dates),
    "raw_missing_calendar_dates": int(len(raw_date_gaps)),
    "processed_missing_calendar_dates": int(
        len(processed_date_gaps)
    ),
    "raw_duplicate_dates": int(raw_duplicate_dates),
    "processed_duplicate_dates": int(processed_duplicate_dates),
    "raw_target_nulls": int(raw_target_nulls),
    "processed_target_nulls": int(processed_target_nulls),
    "checks": {
        key: bool(value)
        for key, value in checks.items()
    },
    "overall_status": "PASSED" if all_passed else "REVIEW REQUIRED"
}

report_path = OUTPUT_DIR / "data_refresh_verification_report.json"

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

check_df = pd.DataFrame([
    {
        "Validation": name,
        "Status": "PASSED" if passed else "FAILED"
    }
    for name, passed in checks.items()
])

check_path = OUTPUT_DIR / "data_refresh_verification_checks.csv"
check_df.to_csv(check_path, index=False)

print("\nFINAL STATUS")
print("=" * 65)

if all_passed:
    print("DATA REFRESH VERIFICATION: PASSED")
else:
    print("DATA REFRESH VERIFICATION: REVIEW REQUIRED")

print("\nReports saved:")
print(report_path)
print(check_path)

display(check_df)

DATA REFRESH VERIFICATION

DATASET INFORMATION
-----------------------------------------------------------------
Raw dataset shape: (1170, 6)
Processed dataset shape: (720, 6)
Raw dataset modified: 2026-11-04 02:54:01.930664
Processed dataset modified: 2026-11-04 13:16:39.987036


ValueError: Missing columns. Raw: ['date', 'children_apprehended_and_placed_in_cbp_custody', 'children_in_cbp_custody', 'children_transferred_out_of_cbp_custody', 'children_in_hhs_care', 'children_discharged_from_hhs_care']; Processed: []

In [82]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "HHS_Unaccompanied_Alien_Children_Program.csv"

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_care_data.csv"

raw_df = pd.read_csv(RAW_PATH)
processed_df = pd.read_csv(PROCESSED_PATH)

print("=" * 65)
print("RAW DATASET COLUMN NAMES")
print("=" * 65)

for i, col in enumerate(raw_df.columns, 1):
    print(f"{i}. {repr(col)}")

print("\nRAW DATASET FIRST FIVE ROWS")
display(raw_df.head())

print("\n" + "=" * 65)
print("PROCESSED DATASET COLUMN NAMES")
print("=" * 65)

for i, col in enumerate(processed_df.columns, 1):
    print(f"{i}. {repr(col)}")

print("\nPROCESSED DATASET FIRST FIVE ROWS")
display(processed_df.head())

RAW DATASET COLUMN NAMES
1. 'Date'
2. 'Children apprehended and placed in CBP custody*'
3. 'Children in CBP custody'
4. 'Children transferred out of CBP custody'
5. 'Children in HHS Care'
6. 'Children discharged from HHS Care'

RAW DATASET FIRST FIVE ROWS


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0



PROCESSED DATASET COLUMN NAMES
1. 'date'
2. 'children_apprehended_and_placed_in_cbp_custody'
3. 'children_in_cbp_custody'
4. 'children_transferred_out_of_cbp_custody'
5. 'children_in_hhs_care'
6. 'children_discharged_from_hhs_care'

PROCESSED DATASET FIRST FIVE ROWS


,date,children_apprehended_and_placed_in_cbp_custody,children_in_cbp_custody,children_transferred_out_of_cbp_custody,children_in_hhs_care,children_discharged_from_hhs_care
0,2023-01-12,33,53,34,6566,436
1,2023-01-22,32,49,39,7122,227
2,2023-01-23,32,50,39,7280,181
3,2023-01-24,47,42,47,7433,175
4,2023-01-25,20,22,41,7538,180


In [84]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import json
import re
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "HHS_Unaccompanied_Alien_Children_Program.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_care_data.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deployment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = OUTPUT_DIR / "data_refresh_verification_report.json"
CHECKS_PATH = OUTPUT_DIR / "data_refresh_verification_checks.csv"
DETAIL_PATH = OUTPUT_DIR / "data_refresh_comparison_details.csv"

EXPECTED_COLUMNS = [
    "date",
    "children_apprehended_and_placed_in_cbp_custody",
    "children_in_cbp_custody",
    "children_transferred_out_of_cbp_custody",
    "children_in_hhs_care",
    "children_discharged_from_hhs_care"
]

COLUMN_ALIASES = {
    "date": [
        "date", "report_date", "data_date", "as_of_date",
        "week_ending", "week", "observation_date"
    ],
    "children_apprehended_and_placed_in_cbp_custody": [
        "children_apprehended_and_placed_in_cbp_custody",
        "children_apprehended_and_placed_in_cbp_custody_at_the_southwest_border",
        "children_apprehended_and_placed_in_cbp_custody_by_cbp",
        "apprehended_and_placed_in_cbp_custody"
    ],
    "children_in_cbp_custody": [
        "children_in_cbp_custody",
        "children_in_cbp_custody_at_the_southwest_border",
        "children_in_cbp_custody_by_cbp"
    ],
    "children_transferred_out_of_cbp_custody": [
        "children_transferred_out_of_cbp_custody",
        "children_transferred_from_cbp_to_hhs",
        "transferred_out_of_cbp_custody"
    ],
    "children_in_hhs_care": [
        "children_in_hhs_care",
        "children_in_hhs_custody",
        "children_in_hhs_care_at_the_end_of_the_reporting_period",
        "children_in_hhs_care_at_the_end_of_the_month"
    ],
    "children_discharged_from_hhs_care": [
        "children_discharged_from_hhs_care",
        "children_discharged_from_hhs_custody",
        "discharged_from_hhs_care"
    ]
}

def normalize_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")

def load_csv_safely(path):
    encodings = ["utf-8-sig", "utf-8", "cp1252", "latin1"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding), encoding
        except Exception as error:
            last_error = error

    raise RuntimeError(f"Unable to read {path}: {last_error}")

def map_columns(df):
    original_columns = list(df.columns)
    normalized = {col: normalize_name(col) for col in original_columns}

    aliases = {}
    for expected, possible_names in COLUMN_ALIASES.items():
        aliases[expected] = {
            normalize_name(expected),
            *[normalize_name(name) for name in possible_names]
        }

    mapping = {}

    for expected, candidates in aliases.items():
        matches = [
            original for original, normalized_name in normalized.items()
            if normalized_name in candidates
        ]

        if len(matches) == 1:
            mapping[matches[0]] = expected

    return df.rename(columns=mapping), mapping

def identify_date_column(df):
    if "date" in df.columns:
        return "date"

    candidates = [
        col for col in df.columns
        if any(term in normalize_name(col) for term in
               ["date", "week", "period", "month", "time"])
    ]

    for col in candidates:
        parsed = pd.to_datetime(df[col], errors="coerce")
        if parsed.notna().mean() >= 0.70:
            return col

    return None

def numeric_series(series):
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")

print("=" * 75)
print("DATA REFRESH VERIFICATION")
print("=" * 75)

for path in [RAW_PATH, PROCESSED_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

raw_original, raw_encoding = load_csv_safely(RAW_PATH)
processed_original, processed_encoding = load_csv_safely(PROCESSED_PATH)

raw_df, raw_mapping = map_columns(raw_original.copy())
processed_df, processed_mapping = map_columns(processed_original.copy())

raw_date_column = identify_date_column(raw_df)
processed_date_column = identify_date_column(processed_df)

if raw_date_column is None:
    raise ValueError(
        "Raw dataset date column could not be identified. "
        f"Actual columns: {list(raw_original.columns)}"
    )

if processed_date_column is None:
    raise ValueError(
        "Processed dataset date column could not be identified."
    )

raw_df = raw_df.rename(columns={raw_date_column: "date"})
processed_df = processed_df.rename(
    columns={processed_date_column: "date"}
)

raw_df["date"] = pd.to_datetime(raw_df["date"], errors="coerce")
processed_df["date"] = pd.to_datetime(
    processed_df["date"], errors="coerce"
)

print("\nRAW DATASET")
print("-" * 75)
print("Shape:", raw_original.shape)
print("Encoding:", raw_encoding)
print("Original columns:", list(raw_original.columns))
print("Mapped columns:", list(raw_df.columns))

print("\nPROCESSED DATASET")
print("-" * 75)
print("Shape:", processed_original.shape)
print("Encoding:", processed_encoding)
print("Columns:", list(processed_df.columns))

raw_missing = [
    col for col in EXPECTED_COLUMNS if col not in raw_df.columns
]
processed_missing = [
    col for col in EXPECTED_COLUMNS if col not in processed_df.columns
]

checks = {}

checks["All required raw columns identified"] = len(raw_missing) == 0
checks["All required processed columns available"] = len(processed_missing) == 0
checks["Raw date values valid"] = raw_df["date"].notna().all()
checks["Processed date values valid"] = processed_df["date"].notna().all()

raw_valid = raw_df.dropna(subset=["date"]).copy()
processed_valid = processed_df.dropna(subset=["date"]).copy()

raw_duplicate_dates = int(raw_valid["date"].duplicated().sum())
processed_duplicate_dates = int(
    processed_valid["date"].duplicated().sum()
)

checks["Raw dataset has no duplicate dates"] = raw_duplicate_dates == 0
checks["Processed dataset has no duplicate dates"] = processed_duplicate_dates == 0

raw_latest = raw_valid["date"].max()
processed_latest = processed_valid["date"].max()
raw_earliest = raw_valid["date"].min()
processed_earliest = processed_valid["date"].min()

checks["Raw and processed latest dates match"] = (
    pd.notna(raw_latest)
    and pd.notna(processed_latest)
    and raw_latest == processed_latest
)

checks["Processed dates fall within raw date coverage"] = (
    pd.notna(raw_earliest)
    and pd.notna(raw_latest)
    and pd.notna(processed_earliest)
    and pd.notna(processed_latest)
    and processed_earliest >= raw_earliest
    and processed_latest <= raw_latest
)

raw_date_set = set(raw_valid["date"].dropna())
processed_date_set = set(processed_valid["date"].dropna())

checks["Every processed date exists in raw dataset"] = (
    processed_date_set.issubset(raw_date_set)
)

raw_calendar_gaps = (
    len(pd.date_range(raw_earliest, raw_latest, freq="D").difference(
        pd.DatetimeIndex(raw_date_set)
    ))
    if pd.notna(raw_earliest) and pd.notna(raw_latest) else None
)

processed_calendar_gaps = (
    len(pd.date_range(processed_earliest, processed_latest, freq="D").difference(
        pd.DatetimeIndex(processed_date_set)
    ))
    if pd.notna(processed_earliest) and pd.notna(processed_latest) else None
)

comparison_rows = []

for col in EXPECTED_COLUMNS:
    if col == "date":
        continue

    if col not in processed_df.columns:
        continue

    processed_df[col] = numeric_series(processed_df[col])

    if col in raw_df.columns:
        raw_df[col] = numeric_series(raw_df[col])

    null_count = int(processed_df[col].isna().sum())

    comparison_rows.append({
        "Column": col,
        "Raw_Available": col in raw_df.columns,
        "Processed_Null_Count": null_count,
        "Processed_Negative_Count": int(
            (processed_df[col].dropna() < 0).sum()
        ),
        "Processed_Minimum": (
            float(processed_df[col].min())
            if processed_df[col].notna().any() else None
        ),
        "Processed_Maximum": (
            float(processed_df[col].max())
            if processed_df[col].notna().any() else None
        )
    })

    checks[f"{col} has no missing processed values"] = null_count == 0
    checks[f"{col} has no negative processed values"] = (
        (processed_df[col].dropna() >= 0).all()
    )

comparison_df = pd.DataFrame(comparison_rows)

if "children_in_hhs_care" in processed_df.columns:
    checks["Primary target column available"] = True
else:
    checks["Primary target column available"] = False

if "children_in_hhs_care" in raw_df.columns:
    raw_latest_target = raw_valid.loc[
        raw_valid["date"] == raw_latest, "children_in_hhs_care"
    ]
    processed_latest_target = processed_valid.loc[
        processed_valid["date"] == processed_latest,
        "children_in_hhs_care"
    ]

    if len(raw_latest_target) and len(processed_latest_target):
        checks["Latest target values match"] = (
            pd.to_numeric(raw_latest_target.iloc[-1], errors="coerce")
            == pd.to_numeric(processed_latest_target.iloc[-1], errors="coerce")
        )
    else:
        checks["Latest target values match"] = False
else:
    checks["Latest target values match"] = False

print("\nDATE COVERAGE")
print("-" * 75)
print("Raw date range:", raw_earliest, "to", raw_latest)
print("Processed date range:", processed_earliest, "to", processed_latest)
print("Raw calendar gaps:", raw_calendar_gaps)
print("Processed calendar gaps:", processed_calendar_gaps)
print("Raw duplicate dates:", raw_duplicate_dates)
print("Processed duplicate dates:", processed_duplicate_dates)

print("\nCOLUMN MAPPING")
print("-" * 75)
print("Raw mapping:", raw_mapping)
print("Missing raw columns:", raw_missing)
print("Missing processed columns:", processed_missing)

print("\nNUMERIC DATA QUALITY")
display(comparison_df)

print("\nVALIDATION RESULTS")
print("-" * 75)

for name, passed in checks.items():
    print(f"{'PASSED' if passed else 'FAILED'}: {name}")

overall_passed = all(checks.values())

report = {
    "verification_timestamp": datetime.now().isoformat(),
    "raw_file": str(RAW_PATH),
    "processed_file": str(PROCESSED_PATH),
    "raw_shape": list(raw_original.shape),
    "processed_shape": list(processed_original.shape),
    "raw_original_columns": [str(c) for c in raw_original.columns],
    "raw_column_mapping": {
        str(k): str(v) for k, v in raw_mapping.items()
    },
    "missing_raw_columns": raw_missing,
    "missing_processed_columns": processed_missing,
    "raw_date_range": [
        str(raw_earliest), str(raw_latest)
    ],
    "processed_date_range": [
        str(processed_earliest), str(processed_latest)
    ],
    "raw_calendar_gaps": raw_calendar_gaps,
    "processed_calendar_gaps": processed_calendar_gaps,
    "raw_duplicate_dates": raw_duplicate_dates,
    "processed_duplicate_dates": processed_duplicate_dates,
    "numeric_quality": comparison_df.to_dict(orient="records"),
    "checks": {k: bool(v) for k, v in checks.items()},
    "overall_status": (
        "PASSED" if overall_passed else "REVIEW REQUIRED"
    )
}

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(report, file, indent=4, default=str)

pd.DataFrame([
    {"Validation": name, "Status": "PASSED" if passed else "FAILED"}
    for name, passed in checks.items()
]).to_csv(CHECKS_PATH, index=False)

comparison_df.to_csv(DETAIL_PATH, index=False)

print("\n" + "=" * 75)
print("FINAL DATA REFRESH VERIFICATION STATUS")
print("=" * 75)
print("PASSED" if overall_passed else "REVIEW REQUIRED")
print("Report:", REPORT_PATH)
print("Checks:", CHECKS_PATH)
print("Details:", DETAIL_PATH)

display(pd.DataFrame([
    {"Validation": name, "Status": "PASSED" if passed else "FAILED"}
    for name, passed in checks.items()
]))

if not overall_passed:
    print(
        "\nSome checks require review. The original datasets were not modified."
    )

DATA REFRESH VERIFICATION

RAW DATASET
---------------------------------------------------------------------------
Shape: (1170, 6)
Encoding: utf-8-sig
Original columns: ['Date', 'Children apprehended and placed in CBP custody*', 'Children in CBP custody', 'Children transferred out of CBP custody', 'Children in HHS Care', 'Children discharged from HHS Care']
Mapped columns: ['date', 'children_apprehended_and_placed_in_cbp_custody', 'children_in_cbp_custody', 'children_transferred_out_of_cbp_custody', 'children_in_hhs_care', 'children_discharged_from_hhs_care']

PROCESSED DATASET
---------------------------------------------------------------------------
Shape: (720, 6)
Encoding: utf-8-sig
Columns: ['date', 'children_apprehended_and_placed_in_cbp_custody', 'children_in_cbp_custody', 'children_transferred_out_of_cbp_custody', 'children_in_hhs_care', 'children_discharged_from_hhs_care']

DATE COVERAGE
---------------------------------------------------------------------------
Raw date ran

,Column,Raw_Available,Processed_Null_Count,Processed_Negative_Count,Processed_Minimum,Processed_Maximum
0,children_apprehended_and_placed_in_cbp_custody,True,0,0,0.0,333.0
1,children_in_cbp_custody,True,0,0,7.0,531.0
2,children_transferred_out_of_cbp_custody,True,0,0,0.0,440.0
3,children_in_hhs_care,True,0,0,1972.0,11516.0
4,children_discharged_from_hhs_care,True,0,0,0.0,505.0



VALIDATION RESULTS
---------------------------------------------------------------------------
PASSED: All required raw columns identified
PASSED: All required processed columns available
FAILED: Raw date values valid
PASSED: Processed date values valid
PASSED: Raw dataset has no duplicate dates
PASSED: Processed dataset has no duplicate dates
PASSED: Raw and processed latest dates match
PASSED: Processed dates fall within raw date coverage
PASSED: Every processed date exists in raw dataset
PASSED: children_apprehended_and_placed_in_cbp_custody has no missing processed values
PASSED: children_apprehended_and_placed_in_cbp_custody has no negative processed values
PASSED: children_in_cbp_custody has no missing processed values
PASSED: children_in_cbp_custody has no negative processed values
PASSED: children_transferred_out_of_cbp_custody has no missing processed values
PASSED: children_transferred_out_of_cbp_custody has no negative processed values
PASSED: children_in_hhs_care has no mi

,Validation,Status
0,All required raw columns identified,PASSED
1,All required processed columns available,PASSED
2,Raw date values valid,FAILED
3,Processed date values valid,PASSED
4,Raw dataset has no duplicate dates,PASSED
5,Processed dataset has no duplicate dates,PASSED
6,Raw and processed latest dates match,PASSED
7,Processed dates fall within raw date coverage,PASSED
8,Every processed date exists in raw dataset,PASSED
9,children_apprehended_and_placed_in_cbp_custody...,PASSED



Some checks require review. The original datasets were not modified.


In [90]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "HHS_Unaccompanied_Alien_Children_Program.csv"

raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")

print("=" * 75)
print("RAW CSV ORIGINAL COLUMN HEADERS")
print("=" * 75)

for i, column in enumerate(raw.columns, 1):
    print(f"{i}. {repr(column)}")

print("\nRAW CSV FIRST FIVE ROWS")
display(raw.head())

print("\nPOSSIBLE DATE COLUMNS")
print("-" * 75)

for column in raw.columns:
    sample = raw[column].dropna().astype(str).head(5).tolist()
    parsed = pd.to_datetime(raw[column], errors="coerce")
    success_rate = round(parsed.notna().mean() * 100, 2)

    if success_rate >= 50 or any(
        word in str(column).lower()
        for word in ["date", "week", "period", "time"]
    ):
        print("Column:", repr(column))
        print("Sample:", sample)
        print("Date parsing success:", success_rate, "%")
        print("-" * 50)

print("\nRAW DATASET SHAPE:", raw.shape)

RAW CSV ORIGINAL COLUMN HEADERS
1. 'Date'
2. 'Children apprehended and placed in CBP custody*'
3. 'Children in CBP custody'
4. 'Children transferred out of CBP custody'
5. 'Children in HHS Care'
6. 'Children discharged from HHS Care'

RAW CSV FIRST FIVE ROWS


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0



POSSIBLE DATE COLUMNS
---------------------------------------------------------------------------
Column: 'Date'
Sample: ['December 21, 2025', 'December 18, 2025', 'December 17, 2025', 'December 16, 2025', 'December 15, 2025']
Date parsing success: 61.54 %
--------------------------------------------------
Column: 'Children apprehended and placed in CBP custody*'
Sample: ['6.0', '11.0', '7.0', '8.0', '11.0']
Date parsing success: 61.54 %
--------------------------------------------------
Column: 'Children in CBP custody'
Sample: ['18.0', '50.0', '31.0', '54.0', '42.0']
Date parsing success: 61.54 %
--------------------------------------------------
Column: 'Children transferred out of CBP custody'
Sample: ['11.0', '6.0', '11.0', '15.0', '9.0']
Date parsing success: 61.54 %
--------------------------------------------------
Column: 'Children discharged from HHS Care'
Sample: ['14.0', '16.0', '10.0', '9.0', '7.0']
Date parsing success: 61.54 %
-------------------------------------------

In [92]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import json
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "HHS_Unaccompanied_Alien_Children_Program.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_care_data.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "deployment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = OUTPUT_DIR / "data_refresh_verification_report.json"
CHECKS_PATH = OUTPUT_DIR / "data_refresh_verification_checks.csv"
DETAIL_PATH = OUTPUT_DIR / "data_refresh_comparison_details.csv"

COLUMN_MAPPING = {
    "Date": "date",
    "Children apprehended and placed in CBP custody*":
        "children_apprehended_and_placed_in_cbp_custody",
    "Children in CBP custody": "children_in_cbp_custody",
    "Children transferred out of CBP custody":
        "children_transferred_out_of_cbp_custody",
    "Children in HHS Care": "children_in_hhs_care",
    "Children discharged from HHS Care":
        "children_discharged_from_hhs_care"
}

EXPECTED_COLUMNS = list(COLUMN_MAPPING.values())
TARGET = "children_in_hhs_care"

print("=" * 75)
print("DATA REFRESH VERIFICATION")
print("=" * 75)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found: {RAW_PATH}")

if not PROCESSED_PATH.exists():
    raise FileNotFoundError(f"Processed dataset not found: {PROCESSED_PATH}")

raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
processed = pd.read_csv(PROCESSED_PATH, encoding="utf-8-sig")

raw.columns = raw.columns.str.strip()
processed.columns = processed.columns.str.strip()

raw_missing_original = [
    col for col in COLUMN_MAPPING if col not in raw.columns
]

processed_missing = [
    col for col in EXPECTED_COLUMNS if col not in processed.columns
]

if raw_missing_original:
    raise ValueError(
        f"Unexpected raw CSV headers: {raw_missing_original}"
    )

if processed_missing:
    raise ValueError(
        f"Unexpected processed CSV headers: {processed_missing}"
    )

raw = raw.rename(columns=COLUMN_MAPPING)

raw["date"] = pd.to_datetime(
    raw["date"].astype(str).str.strip(),
    format="%B %d, %Y",
    errors="coerce"
)

processed["date"] = pd.to_datetime(
    processed["date"],
    errors="coerce"
)

def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan}),
        errors="coerce"
    )

for column in EXPECTED_COLUMNS:
    if column != "date":
        raw[column] = clean_numeric(raw[column])
        processed[column] = clean_numeric(processed[column])

raw_valid = raw.dropna(subset=["date"]).copy()
processed_valid = processed.dropna(subset=["date"]).copy()

raw_latest = raw_valid["date"].max()
processed_latest = processed_valid["date"].max()

raw_earliest = raw_valid["date"].min()
processed_earliest = processed_valid["date"].min()

raw_latest_rows = raw_valid.loc[
    raw_valid["date"] == raw_latest
]

processed_latest_rows = processed_valid.loc[
    processed_valid["date"] == processed_latest
]

raw_latest_values = (
    raw_latest_rows[TARGET].dropna().unique().tolist()
)

processed_latest_values = (
    processed_latest_rows[TARGET].dropna().unique().tolist()
)

if len(raw_latest_values) == 1 and len(processed_latest_values) == 1:
    latest_target_match = (
        raw_latest_values[0] == processed_latest_values[0]
        and raw_latest == processed_latest
    )
else:
    latest_target_match = False

raw_dates = set(raw_valid["date"])
processed_dates = set(processed_valid["date"])

raw_duplicate_dates = int(raw_valid["date"].duplicated().sum())
processed_duplicate_dates = int(
    processed_valid["date"].duplicated().sum()
)

raw_calendar_gaps = len(
    pd.date_range(raw_earliest, raw_latest, freq="D").difference(
        pd.DatetimeIndex(raw_dates)
    )
)

processed_calendar_gaps = len(
    pd.date_range(
        processed_earliest, processed_latest, freq="D"
    ).difference(pd.DatetimeIndex(processed_dates))
)

checks = {
    "All required raw columns identified":
        all(col in raw.columns for col in EXPECTED_COLUMNS),

    "All required processed columns available":
        all(col in processed.columns for col in EXPECTED_COLUMNS),

    "Raw date values valid":
        raw["date"].notna().all(),

    "Processed date values valid":
        processed["date"].notna().all(),

    "Raw dataset has no duplicate dates":
        raw_duplicate_dates == 0,

    "Processed dataset has no duplicate dates":
        processed_duplicate_dates == 0,

    "Raw target values have no missing values":
        raw[TARGET].notna().all(),

    "Processed target values have no missing values":
        processed[TARGET].notna().all(),

    "Processed dates fall within raw date coverage":
        processed_earliest >= raw_earliest
        and processed_latest <= raw_latest,

    "Every processed date exists in raw dataset":
        processed_dates.issubset(raw_dates),

    "Processed target values are non-negative":
        (processed[TARGET].dropna() >= 0).all(),

    "Latest target values match":
        latest_target_match
}

numeric_details = []

for column in EXPECTED_COLUMNS:
    if column == "date":
        continue

    numeric_details.append({
        "Column": column,
        "Raw_Null_Count": int(raw[column].isna().sum()),
        "Processed_Null_Count": int(processed[column].isna().sum()),
        "Raw_Minimum": (
            float(raw[column].min())
            if raw[column].notna().any() else None
        ),
        "Processed_Minimum": (
            float(processed[column].min())
            if processed[column].notna().any() else None
        ),
        "Raw_Maximum": (
            float(raw[column].max())
            if raw[column].notna().any() else None
        ),
        "Processed_Maximum": (
            float(processed[column].max())
            if processed[column].notna().any() else None
        )
    })

details_df = pd.DataFrame(numeric_details)

print("\nDATASET INFORMATION")
print("-" * 75)
print("Raw dataset shape:", raw.shape)
print("Processed dataset shape:", processed.shape)
print("Raw date range:", raw_earliest, "to", raw_latest)
print("Processed date range:", processed_earliest, "to", processed_latest)

print("\nDATA QUALITY")
print("-" * 75)
print("Raw duplicate dates:", raw_duplicate_dates)
print("Processed duplicate dates:", processed_duplicate_dates)
print("Raw calendar gaps:", raw_calendar_gaps)
print("Processed calendar gaps:", processed_calendar_gaps)

print("\nLATEST TARGET VALUE COMPARISON")
print("-" * 75)
print("Raw latest date:", raw_latest)
print("Processed latest date:", processed_latest)
print("Raw latest target values:", raw_latest_values)
print("Processed latest target values:", processed_latest_values)

print("\nALL VALIDATION RESULTS")
print("-" * 75)

for name, passed in checks.items():
    print(f"{'PASSED' if passed else 'FAILED'}: {name}")

print("\nNUMERIC COLUMN DETAILS")
display(details_df)

overall_passed = all(checks.values())

report = {
    "verification_timestamp": datetime.now().isoformat(),
    "raw_dataset": str(RAW_PATH),
    "processed_dataset": str(PROCESSED_PATH),
    "raw_observations": int(len(raw)),
    "processed_observations": int(len(processed)),
    "raw_date_range": [str(raw_earliest), str(raw_latest)],
    "processed_date_range": [
        str(processed_earliest), str(processed_latest)
    ],
    "raw_latest_target_values": raw_latest_values,
    "processed_latest_target_values": processed_latest_values,
    "raw_duplicate_dates": raw_duplicate_dates,
    "processed_duplicate_dates": processed_duplicate_dates,
    "raw_calendar_gaps": raw_calendar_gaps,
    "processed_calendar_gaps": processed_calendar_gaps,
    "checks": {key: bool(value) for key, value in checks.items()},
    "overall_status": (
        "PASSED" if overall_passed else "REVIEW REQUIRED"
    )
}

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(report, file, indent=4, default=str)

checks_df = pd.DataFrame([
    {
        "Validation": name,
        "Status": "PASSED" if passed else "FAILED"
    }
    for name, passed in checks.items()
])

checks_df.to_csv(CHECKS_PATH, index=False)
details_df.to_csv(DETAIL_PATH, index=False)

print("\n" + "=" * 75)
print("FINAL DATA REFRESH VERIFICATION")
print("=" * 75)

print("OVERALL STATUS:", report["overall_status"])
print("Verification report:", REPORT_PATH)
print("Validation checks:", CHECKS_PATH)
print("Numeric details:", DETAIL_PATH)

display(checks_df)

DATA REFRESH VERIFICATION

DATASET INFORMATION
---------------------------------------------------------------------------
Raw dataset shape: (1170, 6)
Processed dataset shape: (720, 6)
Raw date range: 2023-01-12 00:00:00 to 2025-12-21 00:00:00
Processed date range: 2023-01-12 00:00:00 to 2025-12-21 00:00:00

DATA QUALITY
---------------------------------------------------------------------------
Raw duplicate dates: 0
Processed duplicate dates: 0
Raw calendar gaps: 355
Processed calendar gaps: 355

LATEST TARGET VALUE COMPARISON
---------------------------------------------------------------------------
Raw latest date: 2025-12-21 00:00:00
Processed latest date: 2025-12-21 00:00:00
Raw latest target values: [2484.0]
Processed latest target values: [2484]

ALL VALIDATION RESULTS
---------------------------------------------------------------------------
PASSED: All required raw columns identified
PASSED: All required processed columns available
FAILED: Raw date values valid
PASSED: Pro

,Column,Raw_Null_Count,Processed_Null_Count,Raw_Minimum,Processed_Minimum,Raw_Maximum,Processed_Maximum
0,children_apprehended_and_placed_in_cbp_custody,450,0,0.0,0.0,333.0,333.0
1,children_in_cbp_custody,450,0,7.0,7.0,531.0,531.0
2,children_transferred_out_of_cbp_custody,450,0,0.0,0.0,440.0,440.0
3,children_in_hhs_care,450,0,1972.0,1972.0,11516.0,11516.0
4,children_discharged_from_hhs_care,450,0,0.0,0.0,505.0,505.0



FINAL DATA REFRESH VERIFICATION
OVERALL STATUS: REVIEW REQUIRED
Verification report: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\data_refresh_verification_report.json
Validation checks: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\data_refresh_verification_checks.csv
Numeric details: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\data_refresh_comparison_details.csv


,Validation,Status
0,All required raw columns identified,PASSED
1,All required processed columns available,PASSED
2,Raw date values valid,FAILED
3,Processed date values valid,PASSED
4,Raw dataset has no duplicate dates,PASSED
5,Processed dataset has no duplicate dates,PASSED
6,Raw target values have no missing values,FAILED
7,Processed target values have no missing values,PASSED
8,Processed dates fall within raw date coverage,PASSED
9,Every processed date exists in raw dataset,PASSED


In [94]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "HHS_Unaccompanied_Alien_Children_Program.csv"

raw = pd.read_csv(RAW_PATH, dtype=str)

DATE_COLUMN = "Date"
TARGET_COLUMN = "Children in HHS Care"

raw[DATE_COLUMN] = raw[DATE_COLUMN].astype("string").str.strip()

raw["_parsed_date"] = pd.to_datetime(
    raw[DATE_COLUMN],
    format="%B %d, %Y",
    errors="coerce"
)

raw["_parsed_target"] = pd.to_numeric(
    raw[TARGET_COLUMN]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip(),
    errors="coerce"
)

invalid_dates = raw[raw["_parsed_date"].isna()].copy()

missing_targets = raw[
    raw["_parsed_target"].isna()
].copy()

print("=" * 75)
print("RAW DATA QUALITY DIAGNOSTIC")
print("=" * 75)

print("\nDATASET SUMMARY")
print("Total raw rows:", len(raw))
print("Invalid date rows:", len(invalid_dates))
print("Missing or non-numeric target rows:", len(missing_targets))

print("\nINVALID DATE VALUES")
if invalid_dates.empty:
    print("No invalid date values found.")
else:
    print(
        invalid_dates[[DATE_COLUMN]]
        .to_string(index=False)
    )

print("\nMISSING OR NON-NUMERIC TARGET VALUES")
if missing_targets.empty:
    print("No missing or non-numeric target values found.")
else:
    print(
        missing_targets[[DATE_COLUMN, TARGET_COLUMN]]
        .to_string(index=False)
    )

print("\nTARGET VALUES THAT REQUIRE ATTENTION")
print(
    raw.loc[
        raw["_parsed_target"].isna(),
        TARGET_COLUMN
    ].value_counts(dropna=False).to_string()
)

print("\nDATE FORMAT EXAMPLES")
print(
    raw.loc[
        raw["_parsed_date"].isna(),
        DATE_COLUMN
    ].head(20).to_string(index=False)
)

print("\nDIAGNOSTIC COMPLETED")

RAW DATA QUALITY DIAGNOSTIC

DATASET SUMMARY
Total raw rows: 1170
Invalid date rows: 450
Missing or non-numeric target rows: 450

INVALID DATE VALUES
Date
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>
<NA>


In [96]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import json
import hashlib
import joblib
import warnings
import traceback

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "HHS_Unaccompanied_Alien_Children_Program.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_care_data.csv"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
DEPLOYMENT_DIR = OUTPUT_DIR / "deployment"
MONITORING_DIR = OUTPUT_DIR / "monitoring"
DOCUMENTATION_DIR = PROJECT_ROOT / "documentation"

for folder in [DEPLOYMENT_DIR, MONITORING_DIR, DOCUMENTATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET = "children_in_hhs_care"
DATE_COLUMN = "date"

RAW_DATE_COLUMN = "Date"
RAW_TARGET_COLUMN = "Children in HHS Care"

COLUMN_MAPPING = {
    "Date": "date",
    "Children apprehended and placed in CBP custody*": "children_apprehended_and_placed_in_cbp_custody",
    "Children in CBP custody": "children_in_cbp_custody",
    "Children transferred out of CBP custody": "children_transferred_out_of_cbp_custody",
    "Children in HHS Care": "children_in_hhs_care",
    "Children discharged from HHS Care": "children_discharged_from_hhs_care"
}

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

results = []
model_versions = []
reproducibility_results = []
monitoring_results = []
drift_results = []
refresh_details = []

def add_result(area, check, status, details):
    results.append({
        "Area": area,
        "Check": check,
        "Status": status,
        "Details": str(details)
    })

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, default=str)

def calculate_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def calculate_psi(reference, current, bins=10):
    reference = pd.to_numeric(pd.Series(reference), errors="coerce").dropna()
    current = pd.to_numeric(pd.Series(current), errors="coerce").dropna()

    if len(reference) < 10 or len(current) < 10:
        return np.nan

    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(reference, quantiles))

    if len(edges) < 3:
        return 0.0

    edges[0] = -np.inf
    edges[-1] = np.inf

    ref_counts = np.histogram(reference, bins=edges)[0]
    cur_counts = np.histogram(current, bins=edges)[0]

    ref_pct = ref_counts / max(ref_counts.sum(), 1)
    cur_pct = cur_counts / max(cur_counts.sum(), 1)

    ref_pct = np.where(ref_pct == 0, 0.0001, ref_pct)
    cur_pct = np.where(cur_pct == 0, 0.0001, cur_pct)

    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

print("=" * 75)
print("FORECAST REPRODUCIBILITY, DATA REFRESH AND MODEL GOVERNANCE")
print("=" * 75)

# Load and validate datasets

raw = pd.DataFrame()
processed = pd.DataFrame()

if RAW_PATH.exists():
    raw = pd.read_csv(RAW_PATH, dtype=str)
    add_result("Data Refresh", "Raw dataset loaded", "PASSED", f"{len(raw)} rows")
else:
    add_result("Data Refresh", "Raw dataset loaded", "FAILED", "Raw dataset not found")

if PROCESSED_PATH.exists():
    processed = pd.read_csv(PROCESSED_PATH)
    add_result("Data Refresh", "Processed dataset loaded", "PASSED", f"{len(processed)} rows")
else:
    add_result("Data Refresh", "Processed dataset loaded", "FAILED", "Processed dataset not found")

# Data refresh verification

if not raw.empty and not processed.empty:

    missing_raw_columns = [
        column for column in COLUMN_MAPPING
        if column not in raw.columns
    ]

    missing_processed_columns = [
        column for column in COLUMN_MAPPING.values()
        if column not in processed.columns
    ]

    add_result(
        "Data Refresh",
        "Required raw columns",
        "PASSED" if not missing_raw_columns else "FAILED",
        missing_raw_columns if missing_raw_columns else "All required raw columns available"
    )

    add_result(
        "Data Refresh",
        "Required processed columns",
        "PASSED" if not missing_processed_columns else "FAILED",
        missing_processed_columns if missing_processed_columns else "All required processed columns available"
    )

    if not missing_raw_columns and not missing_processed_columns:

        raw["_parsed_date"] = pd.to_datetime(
            raw[RAW_DATE_COLUMN].astype("string").str.strip(),
            format="%B %d, %Y",
            errors="coerce"
        )

        processed["_parsed_date"] = pd.to_datetime(
            processed[DATE_COLUMN],
            errors="coerce"
        )

        raw["_parsed_target"] = pd.to_numeric(
            raw[RAW_TARGET_COLUMN]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.strip(),
            errors="coerce"
        )

        processed["_parsed_target"] = pd.to_numeric(
            processed[TARGET],
            errors="coerce"
        )

        invalid_raw_dates = raw[raw["_parsed_date"].isna()]
        missing_raw_targets = raw[raw["_parsed_target"].isna()]

        duplicate_raw_dates = raw[
            raw["_parsed_date"].notna()
        ]["_parsed_date"].duplicated().sum()

        duplicate_processed_dates = processed[
            processed["_parsed_date"].notna()
        ]["_parsed_date"].duplicated().sum()

        add_result(
            "Data Refresh",
            "Raw date values valid",
            "PASSED" if invalid_raw_dates.empty else "FAILED",
            f"{len(invalid_raw_dates)} invalid date rows"
        )

        add_result(
            "Data Refresh",
            "Raw target values complete",
            "PASSED" if missing_raw_targets.empty else "FAILED",
            f"{len(missing_raw_targets)} missing or non-numeric target rows"
        )

        add_result(
            "Data Refresh",
            "Processed date values valid",
            "PASSED" if processed["_parsed_date"].notna().all() else "FAILED",
            f"{processed['_parsed_date'].isna().sum()} invalid processed dates"
        )

        add_result(
            "Data Refresh",
            "Raw duplicate dates",
            "PASSED" if duplicate_raw_dates == 0 else "FAILED",
            f"{duplicate_raw_dates} duplicate dates"
        )

        add_result(
            "Data Refresh",
            "Processed duplicate dates",
            "PASSED" if duplicate_processed_dates == 0 else "FAILED",
            f"{duplicate_processed_dates} duplicate dates"
        )

        add_result(
            "Data Refresh",
            "Processed target completeness",
            "PASSED" if processed["_parsed_target"].notna().all() else "FAILED",
            f"{processed['_parsed_target'].isna().sum()} missing processed targets"
        )

        valid_raw_dates = raw["_parsed_date"].dropna()
        valid_processed_dates = processed["_parsed_date"].dropna()

        if len(valid_raw_dates) and len(valid_processed_dates):

            raw_min = valid_raw_dates.min()
            raw_max = valid_raw_dates.max()
            processed_min = valid_processed_dates.min()
            processed_max = valid_processed_dates.max()

            coverage_ok = (
                processed_min >= raw_min and
                processed_max <= raw_max
            )

            raw_date_set = set(valid_raw_dates)
            processed_date_set = set(valid_processed_dates)

            all_dates_present = processed_date_set.issubset(raw_date_set)

            add_result(
                "Data Refresh",
                "Processed dates within raw coverage",
                "PASSED" if coverage_ok else "FAILED",
                f"Raw: {raw_min.date()} to {raw_max.date()}; Processed: {processed_min.date()} to {processed_max.date()}"
            )

            add_result(
                "Data Refresh",
                "Every processed date exists in raw data",
                "PASSED" if all_dates_present else "FAILED",
                f"{len(processed_date_set - raw_date_set)} processed dates not found in raw data"
            )

            latest_raw = raw[
                raw["_parsed_date"] == raw["_parsed_date"].max()
            ]

            latest_processed = processed[
                processed["_parsed_date"] == processed["_parsed_date"].max()
            ]

            latest_raw_value = latest_raw["_parsed_target"].dropna()
            latest_processed_value = latest_processed["_parsed_target"].dropna()

            latest_match = (
                len(latest_raw_value) == 1 and
                len(latest_processed_value) == 1 and
                latest_raw["_parsed_date"].max() == latest_processed["_parsed_date"].max() and
                np.isclose(
                    latest_raw_value.iloc[0],
                    latest_processed_value.iloc[0]
                )
            )

            add_result(
                "Data Refresh",
                "Latest target values match",
                "PASSED" if latest_match else "FAILED",
                f"Raw={latest_raw_value.tolist()}, Processed={latest_processed_value.tolist()}"
            )

        else:
            add_result(
                "Data Refresh",
                "Date coverage and latest value comparison",
                "NOT VERIFIED",
                "Insufficient valid dates"
            )

        non_negative = (
            processed["_parsed_target"].dropna().ge(0).all()
        )

        add_result(
            "Data Refresh",
            "Processed target non-negative",
            "PASSED" if non_negative else "FAILED",
            "Processed target values checked"
        )

        for index, row in invalid_raw_dates.iterrows():
            refresh_details.append({
                "Issue": "Invalid raw date",
                "CSV Row": index + 2,
                "Date Value": row[RAW_DATE_COLUMN],
                "Target Value": row[RAW_TARGET_COLUMN]
            })

        for index, row in missing_raw_targets.iterrows():
            refresh_details.append({
                "Issue": "Missing or non-numeric raw target",
                "CSV Row": index + 2,
                "Date Value": row[RAW_DATE_COLUMN],
                "Target Value": row[RAW_TARGET_COLUMN]
            })

# Forecast reproducibility testing

MODEL_SEARCH_DIRS = [
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "outputs"
]

excluded_parts = {
    ".venv", ".venv_deployment", ".deployment_test_env",
    "__pycache__", ".ipynb_checkpoints", "site-packages"
}

model_files = []

for search_dir in MODEL_SEARCH_DIRS:
    if search_dir.exists():
        for path in search_dir.rglob("*"):
            if (
                path.is_file()
                and path.suffix.lower() in [".joblib", ".pkl"]
                and not any(part in excluded_parts for part in path.parts)
            ):
                model_files.append(path)

model_files = sorted(set(model_files))

if not model_files:
    add_result(
        "Reproducibility",
        "Model artifact discovery",
        "NOT VERIFIED",
        "No joblib or pickle model artifacts found"
    )

for model_path in model_files:

    model_name = model_path.stem

    try:
        model = joblib.load(model_path)

        prediction_1 = None
        prediction_2 = None
        test_method = None

        if hasattr(model, "feature_names_in_") and hasattr(model, "predict"):

            feature_names = list(model.feature_names_in_)

            test_input = pd.DataFrame(
                np.zeros((1, len(feature_names))),
                columns=feature_names
            )

            prediction_1 = np.asarray(model.predict(test_input))
            prediction_2 = np.asarray(model.predict(test_input))
            test_method = "Repeated prediction using model feature schema"

        elif hasattr(model, "get_forecast"):

            prediction_1 = np.asarray(
                model.get_forecast(steps=30).predicted_mean
            )

            prediction_2 = np.asarray(
                model.get_forecast(steps=30).predicted_mean
            )

            test_method = "Repeated 30-step statistical forecast"

        elif hasattr(model, "predict"):

            try:
                prediction_1 = np.asarray(model.predict())
                prediction_2 = np.asarray(model.predict())
                test_method = "Repeated default prediction"
            except Exception:
                test_method = "No compatible prediction input available"

        if prediction_1 is not None and prediction_2 is not None:

            same = (
                prediction_1.shape == prediction_2.shape
                and np.allclose(
                    prediction_1,
                    prediction_2,
                    rtol=1e-9,
                    atol=1e-9,
                    equal_nan=True
                )
            )

            status = "PASSED" if same else "FAILED"

            reproducibility_results.append({
                "Model": model_name,
                "Status": status,
                "Method": test_method,
                "Prediction Shape": str(prediction_1.shape),
                "Model File": str(model_path)
            })

        else:
            reproducibility_results.append({
                "Model": model_name,
                "Status": "NOT VERIFIED",
                "Method": test_method or "Compatible prediction method unavailable",
                "Prediction Shape": "",
                "Model File": str(model_path)
            })

    except Exception as error:
        reproducibility_results.append({
            "Model": model_name,
            "Status": "FAILED",
            "Method": "Model loading or prediction error",
            "Prediction Shape": "",
            "Model File": str(model_path),
            "Error": str(error)
        })

for item in reproducibility_results:
    add_result(
        "Reproducibility",
        f"Model reproducibility: {item['Model']}",
        item["Status"],
        item.get("Method", "")
    )

if reproducibility_results:
    reproducibility_passed = all(
        item["Status"] == "PASSED"
        for item in reproducibility_results
    )

    add_result(
        "Reproducibility",
        "Overall model inference reproducibility",
        "PASSED" if reproducibility_passed else "NOT VERIFIED",
        f"{sum(x['Status'] == 'PASSED' for x in reproducibility_results)} passed out of {len(reproducibility_results)} model artifacts"
    )

# Model monitoring and drift monitoring

if not processed.empty and "_parsed_target" in processed.columns:

    monitoring_data = processed[
        ["_parsed_date", "_parsed_target"]
    ].dropna().sort_values("_parsed_date")

    if len(monitoring_data) >= 30:

        recent = monitoring_data.tail(30)["_parsed_target"]
        baseline = monitoring_data.iloc[:-30]["_parsed_target"]

        actual_mean = float(recent.mean())
        baseline_mean = float(baseline.mean())

        recent_std = float(recent.std())
        baseline_std = float(baseline.std())

        mean_change_pct = (
            abs(actual_mean - baseline_mean) /
            max(abs(baseline_mean), 1)
        ) * 100

        monitoring_results = [
            {
                "Metric": "Recent observation count",
                "Value": len(recent),
                "Interpretation": "Latest 30 observations"
            },
            {
                "Metric": "Baseline observation count",
                "Value": len(baseline),
                "Interpretation": "All preceding observations"
            },
            {
                "Metric": "Recent target mean",
                "Value": actual_mean,
                "Interpretation": "Latest 30 observations"
            },
            {
                "Metric": "Baseline target mean",
                "Value": baseline_mean,
                "Interpretation": "Historical observations"
            },
            {
                "Metric": "Target mean change percentage",
                "Value": mean_change_pct,
                "Interpretation": "Change in recent mean against baseline"
            },
            {
                "Metric": "Recent target standard deviation",
                "Value": recent_std,
                "Interpretation": "Recent variability"
            },
            {
                "Metric": "Baseline target standard deviation",
                "Value": baseline_std,
                "Interpretation": "Historical variability"
            }
        ]

        add_result(
            "Model Monitoring",
            "Target monitoring statistics",
            "PASSED",
            "Monitoring statistics calculated"
        )

        psi = calculate_psi(baseline, recent)

        if np.isnan(psi):
            drift_status = "NOT VERIFIED"
            drift_interpretation = "Insufficient data for PSI calculation"
        elif psi < 0.10:
            drift_status = "PASSED"
            drift_interpretation = "No significant distribution drift detected"
        elif psi < 0.25:
            drift_status = "WARNING"
            drift_interpretation = "Moderate distribution drift; investigate"
        else:
            drift_status = "ALERT"
            drift_interpretation = "Significant distribution drift; review model"

        drift_results.append({
            "Feature": TARGET,
            "PSI": psi,
            "Status": drift_status,
            "Interpretation": drift_interpretation
        })

        add_result(
            "Drift Monitoring",
            "Target distribution drift",
            drift_status,
            drift_interpretation
        )

        if mean_change_pct >= 20:
            monitoring_alert = "REVIEW REQUIRED"
        else:
            monitoring_alert = "NORMAL"

        add_result(
            "Model Monitoring",
            "Target mean change alert",
            "WARNING" if monitoring_alert == "REVIEW REQUIRED" else "PASSED",
            f"{mean_change_pct:.2f}% change; review threshold is 20%"
        )

    else:
        add_result(
            "Model Monitoring",
            "Monitoring statistics",
            "NOT VERIFIED",
            "At least 30 valid observations are required"
        )

else:
    add_result(
        "Model Monitoring",
        "Monitoring statistics",
        "NOT VERIFIED",
        "Processed target data unavailable"
    )

# Model version documentation

for model_path in model_files:

    try:
        stat = model_path.stat()

        model_versions.append({
            "Model Name": model_path.stem,
            "File Name": model_path.name,
            "Relative Path": str(model_path.relative_to(PROJECT_ROOT)),
            "File Extension": model_path.suffix,
            "File Size Bytes": stat.st_size,
            "Last Modified": datetime.fromtimestamp(stat.st_mtime).isoformat(),
            "SHA256": calculate_sha256(model_path),
            "Version Identifier": calculate_sha256(model_path)[:12]
        })

    except Exception as error:
        add_result(
            "Model Versioning",
            f"Version metadata: {model_path.name}",
            "FAILED",
            error
        )

add_result(
    "Model Versioning",
    "Model artifact inventory",
    "PASSED" if model_versions else "NOT VERIFIED",
    f"{len(model_versions)} model artifacts inventoried"
)

registry_path = PROJECT_ROOT / "models" / "registry" / "model_registry.json"

if registry_path.exists():
    try:
        with open(registry_path, "r", encoding="utf-8") as file:
            registry_data = json.load(file)

        add_result(
            "Model Versioning",
            "Existing model registry readable",
            "PASSED",
            "Model registry JSON loaded successfully"
        )

    except Exception as error:
        registry_data = {}
        add_result(
            "Model Versioning",
            "Existing model registry readable",
            "FAILED",
            error
        )
else:
    registry_data = {}
    add_result(
        "Model Versioning",
        "Existing model registry",
        "NOT VERIFIED",
        "No existing model registry found"
    )

# Retraining criteria

retraining_criteria = {
    "review_frequency": "Monthly, or after a material data or operational change",
    "performance_trigger": {
        "description": "Retrain or investigate when validation MAE or RMSE worsens materially against the approved baseline",
        "suggested_threshold": "20% deterioration for two consecutive evaluation periods"
    },
    "drift_trigger": {
        "description": "Investigate model retraining when PSI indicates significant feature or target distribution drift",
        "suggested_threshold": "PSI >= 0.25"
    },
    "data_quality_trigger": {
        "description": "Investigate before retraining if date gaps, duplicate dates, missing target values, or invalid records appear",
        "action": "Resolve data quality issues before model training"
    },
    "forecast_stability_trigger": {
        "description": "Review the model when forecast outputs become unstable or implausible",
        "action": "Compare against persistence and moving-average baselines"
    },
    "retraining_validation": [
        "Use chronological train-validation-test splits",
        "Use walk-forward validation",
        "Compare against the currently deployed model",
        "Evaluate MAE, RMSE and MAPE where appropriate",
        "Check forecast horizons of 1, 7, 14 and 30 days",
        "Review confidence intervals and operational capacity implications",
        "Promote a replacement only after validation and approval"
    ],
    "deployment_rule": "Never replace the production model automatically based only on a drift alert."
}

add_result(
    "Retraining",
    "Retraining criteria documented",
    "PASSED",
    "Performance, drift, data quality and stability triggers documented"
)

# Generate monitoring and governance documentation

monitoring_document = f"""
# Model Monitoring and Maintenance Procedure

Generated: {timestamp}

## Monitoring scope

Target variable: {TARGET}

Monitoring should cover:
- Forecast accuracy using MAE, RMSE and MAPE.
- Forecast errors by model and forecast horizon.
- Prediction stability and implausible outputs.
- Data completeness, date continuity and duplicate dates.
- Changes in target distribution.
- Model artifact integrity and version identifiers.

## Monitoring frequency

- Data quality: whenever new data is refreshed.
- Forecast performance: after each evaluation cycle.
- Drift indicators: monthly or after a material data change.
- Model review: monthly and whenever an alert occurs.

## Drift interpretation

- PSI below 0.10: no significant drift indicated.
- PSI from 0.10 to below 0.25: investigate moderate drift.
- PSI 0.25 or higher: significant drift alert.

PSI is a screening indicator, not proof that a model has become inaccurate.

## Retraining criteria

A model review is recommended when:
- Validation MAE or RMSE deteriorates by approximately 20% for two consecutive evaluation periods.
- Significant distribution drift is detected.
- Forecast stability deteriorates.
- Data collection or operational definitions change.

Data quality problems must be resolved before retraining.

## Deployment controls

- Preserve the existing production model until a replacement is validated.
- Record model file hashes and version identifiers.
- Keep chronological validation and test results.
- Compare candidate models with the deployed model and baseline models.
- Record approval and deployment dates.
- Retain rollback capability.

## Important limitation

This document defines the monitoring procedure. It does not establish that continuous automated monitoring or scheduled retraining is running.
"""

retraining_document = "# Retraining Criteria\n\n" + json.dumps(
    retraining_criteria,
    indent=4
)

version_document = {
    "generated_at": timestamp,
    "project": PROJECT_ROOT.name,
    "target": TARGET,
    "model_artifacts": model_versions,
    "existing_registry_path": str(registry_path),
    "registry_found": registry_path.exists(),
    "versioning_method": "SHA-256 content hash",
    "note": "Version identifiers are content-based fingerprints, not claims of semantic model release versions."
}

with open(
    DOCUMENTATION_DIR / "MODEL_MONITORING_AND_MAINTENANCE.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(monitoring_document)

with open(
    DOCUMENTATION_DIR / "MODEL_RETRAINING_CRITERIA.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(retraining_document)

save_json(
    version_document,
    DOCUMENTATION_DIR / "MODEL_VERSION_DOCUMENTATION.json"
)

save_json(
    retraining_criteria,
    DOCUMENTATION_DIR / "MODEL_RETRAINING_CRITERIA.json"
)

# Save verification reports

results_df = pd.DataFrame(results)

results_df.to_csv(
    DEPLOYMENT_DIR / "reproducibility_refresh_governance_checks.csv",
    index=False
)

pd.DataFrame(reproducibility_results).to_csv(
    DEPLOYMENT_DIR / "forecast_reproducibility_results.csv",
    index=False
)

pd.DataFrame(refresh_details).to_csv(
    DEPLOYMENT_DIR / "data_refresh_exception_details.csv",
    index=False
)

pd.DataFrame(model_versions).to_csv(
    DEPLOYMENT_DIR / "model_version_inventory.csv",
    index=False
)

pd.DataFrame(monitoring_results).to_csv(
    MONITORING_DIR / "model_monitoring_metrics.csv",
    index=False
)

pd.DataFrame(drift_results).to_csv(
    MONITORING_DIR / "model_drift_report.csv",
    index=False
)

# Save overall report

status_counts = results_df["Status"].value_counts().to_dict()

overall_report = {
    "project": PROJECT_ROOT.name,
    "generated_at": timestamp,
    "run_id": run_id,
    "target": TARGET,
    "model_artifact_count": len(model_files),
    "reproducibility_tests": len(reproducibility_results),
    "model_versions_documented": len(model_versions),
    "checks": len(results_df),
    "status_counts": status_counts,
    "passed_checks": int((results_df["Status"] == "PASSED").sum()),
    "failed_checks": int((results_df["Status"] == "FAILED").sum()),
    "warning_checks": int((results_df["Status"] == "WARNING").sum()),
    "not_verified_checks": int((results_df["Status"] == "NOT VERIFIED").sum()),
    "data_refresh_exceptions": len(refresh_details),
    "overall_status": (
        "PASSED"
        if (
            (results_df["Status"] == "FAILED").sum() == 0
            and (results_df["Status"] == "NOT VERIFIED").sum() == 0
            and (results_df["Status"] == "WARNING").sum() == 0
        )
        else "REVIEW REQUIRED"
    ),
    "important_note": (
        "Repeated inference checks use available model interfaces. "
        "They do not replace a full end-to-end replay of the original training and forecasting pipeline. "
        "Monitoring and retraining documentation does not mean scheduled automation is active."
    )
}

save_json(
    overall_report,
    DEPLOYMENT_DIR / "reproducibility_refresh_governance_report.json"
)

print("\n" + "=" * 75)
print("VALIDATION RESULTS")
print("=" * 75)

print(results_df.to_string(index=False))

print("\n" + "=" * 75)
print("SUMMARY")
print("=" * 75)

print("Total checks:", len(results_df))
print("PASSED:", overall_report["passed_checks"])
print("FAILED:", overall_report["failed_checks"])
print("WARNING:", overall_report["warning_checks"])
print("NOT VERIFIED:", overall_report["not_verified_checks"])
print("Model artifacts discovered:", len(model_files))
print("Reproducibility tests:", len(reproducibility_results))
print("Model versions documented:", len(model_versions))
print("Data refresh exceptions:", len(refresh_details))
print("Overall status:", overall_report["overall_status"])

print("\nGENERATED DOCUMENTATION")
print(DOCUMENTATION_DIR / "MODEL_MONITORING_AND_MAINTENANCE.md")
print(DOCUMENTATION_DIR / "MODEL_RETRAINING_CRITERIA.md")
print(DOCUMENTATION_DIR / "MODEL_VERSION_DOCUMENTATION.json")

print("\nGENERATED REPORTS")
print(DEPLOYMENT_DIR / "reproducibility_refresh_governance_report.json")
print(DEPLOYMENT_DIR / "reproducibility_refresh_governance_checks.csv")
print(DEPLOYMENT_DIR / "forecast_reproducibility_results.csv")
print(DEPLOYMENT_DIR / "data_refresh_exception_details.csv")
print(DEPLOYMENT_DIR / "model_version_inventory.csv")
print(MONITORING_DIR / "model_monitoring_metrics.csv")
print(MONITORING_DIR / "model_drift_report.csv")

print("\nVALIDATION AND DOCUMENTATION RUN COMPLETED")

FORECAST REPRODUCIBILITY, DATA REFRESH AND MODEL GOVERNANCE

VALIDATION RESULTS
            Area                                                          Check  Status                                                            Details
    Data Refresh                                             Raw dataset loaded  PASSED                                                          1170 rows
    Data Refresh                                       Processed dataset loaded  PASSED                                                           720 rows
    Data Refresh                                           Required raw columns  PASSED                                 All required raw columns available
    Data Refresh                                     Required processed columns  PASSED                           All required processed columns available
    Data Refresh                                          Raw date values valid  FAILED                                              450 invalid 

In [98]:
from pathlib import Path
import json

PROJECT_ROOT = Path(
    r"C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand"
)

REPORT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "deployment"
    / "reproducibility_refresh_governance_report.json"
)

print("=" * 65)
print("FINAL DEPLOYMENT VALIDATION STATUS")
print("=" * 65)

if not REPORT_PATH.exists():
    print("REPORT NOT FOUND")
    print("Expected location:", REPORT_PATH)
else:
    with open(REPORT_PATH, "r", encoding="utf-8") as file:
        report = json.load(file)

    print("OVERALL STATUS :", report.get("overall_status", "NOT AVAILABLE"))
    print("TOTAL CHECKS   :", report.get("checks", 0))
    print("PASSED         :", report.get("passed_checks", 0))
    print("FAILED         :", report.get("failed_checks", 0))
    print("WARNING        :", report.get("warning_checks", 0))
    print("NOT VERIFIED   :", report.get("not_verified_checks", 0))

    print("\n" + "=" * 65)

    if report.get("overall_status") == "PASSED":
        print("ALL VALIDATION CHECKS PASSED")
        print("Deployment and maintenance validation completed.")
    else:
        print("VALIDATION REVIEW REQUIRED")
        print("Some checks still need attention.")

    print("=" * 65)

FINAL DEPLOYMENT VALIDATION STATUS
OVERALL STATUS : REVIEW REQUIRED
TOTAL CHECKS   : 41
PASSED         : 37
FAILED         : 2
NOT VERIFIED   : 0

VALIDATION REVIEW REQUIRED
Some checks still need attention.


In [110]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT_ROOT = Path.cwd().resolve()

for folder in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (folder / "data" / "raw" / "HHS_Unaccompanied_Alien_Children_Program.csv").exists():
        PROJECT_ROOT = folder
        break
else:
    raise FileNotFoundError("Project root not found.")

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "HHS_Unaccompanied_Alien_Children_Program.csv"
PROCESSED_FILE = PROJECT_ROOT / "data" / "processed" / "cleaned_care_data.csv"
DEPLOYMENT_DIR = PROJECT_ROOT / "outputs" / "deployment"
MONITORING_DIR = PROJECT_ROOT / "outputs" / "monitoring"
DOCUMENTATION_DIR = PROJECT_ROOT / "documentation"

DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)

def read_csv(path):
    if not path.exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception:
        return None

def read_text(path):
    if not path.exists():
        return ""
    return path.read_text(encoding="utf-8", errors="ignore")

def find_column(df, candidates):
    for candidate in candidates:
        for column in df.columns:
            if candidate in str(column).lower():
                return column
    return None

def normalize_status(value):
    value = str(value).strip().lower()
    if value in ["pass", "passed", "success", "successful", "true", "yes", "verified", "ok"]:
        return "PASS"
    if value in ["fail", "failed", "failure", "false", "error", "no"]:
        return "FAIL"
    return "NOT VERIFIED"

results = []

def record(name, status, details):
    results.append({
        "Verification": name,
        "Status": status,
        "Details": str(details)
    })

# Forecast reproducibility testing
forecast_file = DEPLOYMENT_DIR / "forecast_reproducibility_results.csv"
forecast_df = read_csv(forecast_file)

if forecast_df is None or forecast_df.empty:
    record("Forecast reproducibility testing", "NOT VERIFIED", "Reproducibility report missing or empty.")
else:
    status_col = find_column(forecast_df, ["status", "result", "passed", "success"])

    if status_col is None:
        record("Forecast reproducibility testing", "NOT VERIFIED", "Status column not found.")
    else:
        statuses = forecast_df[status_col].dropna().map(normalize_status)

        if statuses.empty:
            record("Forecast reproducibility testing", "NOT VERIFIED", "No verification results found.")
        elif (statuses == "FAIL").any():
            record("Forecast reproducibility testing", "FAIL", f"{(statuses == 'FAIL').sum()} failed checks.")
        elif (statuses == "NOT VERIFIED").any():
            record("Forecast reproducibility testing", "NOT VERIFIED", "Unresolved reproducibility checks found.")
        else:
            record("Forecast reproducibility testing", "PASS", f"{len(statuses)} checks passed.")

# Data refresh procedure verification
raw = pd.read_csv(RAW_FILE, dtype=str)
raw.columns = raw.columns.str.replace("\ufeff", "", regex=False).str.strip()

raw = raw.dropna(how="all").copy()

raw["Date"] = pd.to_datetime(
    raw["Date"].astype("string").str.strip(),
    format="%B %d, %Y",
    errors="coerce"
)

raw["Children in HHS Care"] = pd.to_numeric(
    raw["Children in HHS Care"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

raw_date_valid = raw["Date"].notna().all()
raw_target_valid = raw["Children in HHS Care"].notna().all()
raw_duplicates_valid = not raw["Date"].duplicated().any()

processed = pd.read_csv(PROCESSED_FILE)

processed["date"] = pd.to_datetime(processed["date"], errors="coerce")
processed["children_in_hhs_care"] = pd.to_numeric(
    processed["children_in_hhs_care"], errors="coerce"
)

processed_valid = (
    processed["date"].notna().all()
    and processed["children_in_hhs_care"].notna().all()
    and not processed["date"].duplicated().any()
)

raw_comparison = raw[["Date", "Children in HHS Care"]].rename(
    columns={
        "Date": "date",
        "Children in HHS Care": "children_in_hhs_care"
    }
).sort_values("date").reset_index(drop=True)

processed_comparison = processed[
    ["date", "children_in_hhs_care"]
].sort_values("date").reset_index(drop=True)

record_counts_match = len(raw_comparison) == len(processed_comparison)

all_records_match = False

if record_counts_match and processed_valid:
    all_records_match = (
        raw_comparison["date"].equals(processed_comparison["date"])
        and np.allclose(
            raw_comparison["children_in_hhs_care"].to_numpy(dtype=float),
            processed_comparison["children_in_hhs_care"].to_numpy(dtype=float),
            equal_nan=False
        )
    )

refresh_checks = {
    "Raw date values valid": raw_date_valid,
    "Raw target values complete": raw_target_valid,
    "Raw dates contain no duplicates": raw_duplicates_valid,
    "Processed dataset valid": processed_valid,
    "Raw and processed record counts match": record_counts_match,
    "All raw and processed records match": all_records_match
}

refresh_pass = all(refresh_checks.values())

refresh_details = "; ".join(
    f"{name}: {'PASS' if status else 'FAIL'}"
    for name, status in refresh_checks.items()
)

record(
    "Data refresh procedure",
    "PASS" if refresh_pass else "FAIL",
    f"{len(raw)} non-empty raw records; {refresh_details}"
)

refresh_report = pd.DataFrame([
    {"Check": name, "Status": "PASS" if status else "FAIL"}
    for name, status in refresh_checks.items()
])

refresh_report.to_csv(
    DEPLOYMENT_DIR / "data_refresh_final_verification.csv",
    index=False
)

# Update data refresh governance report
governance_path = DEPLOYMENT_DIR / "reproducibility_refresh_governance_checks.csv"
governance = read_csv(governance_path)

if governance is not None and not governance.empty:
    name_col = find_column(governance, ["check", "test", "name", "validation"])
    status_col = find_column(governance, ["status", "result", "passed", "success"])

    if name_col is not None and status_col is not None:
        for check_name, check_status in refresh_checks.items():
            mask = governance[name_col].astype(str).str.contains(
                check_name,
                case=False,
                regex=False,
                na=False
            )

            if mask.any():
                governance.loc[mask, status_col] = "PASS" if check_status else "FAIL"

        governance.to_csv(governance_path, index=False)

# Model monitoring procedure verification
monitoring_metrics = read_csv(
    MONITORING_DIR / "model_monitoring_metrics.csv"
)
monitoring_documentation = read_text(
    DOCUMENTATION_DIR / "MODEL_MONITORING_AND_MAINTENANCE.md"
)

if (
    monitoring_metrics is not None
    and not monitoring_metrics.empty
    and len(monitoring_documentation.strip()) > 100
):
    record(
        "Model monitoring procedure",
        "PASS",
        f"Monitoring metrics contain {len(monitoring_metrics)} rows; documentation exists."
    )
else:
    record(
        "Model monitoring procedure",
        "NOT VERIFIED",
        "Monitoring metrics or documentation missing or incomplete."
    )

# Model drift monitoring verification
drift_report = read_csv(
    MONITORING_DIR / "model_drift_report.csv"
)

if drift_report is None or drift_report.empty:
    record("Model drift monitoring", "NOT VERIFIED", "Drift report missing or empty.")
else:
    psi_col = find_column(drift_report, ["psi"])

    if psi_col is None:
        record("Model drift monitoring", "NOT VERIFIED", "PSI column missing.")
    else:
        psi_values = pd.to_numeric(drift_report[psi_col], errors="coerce").dropna()

        if psi_values.empty:
            record("Model drift monitoring", "NOT VERIFIED", "No valid PSI values found.")
        else:
            record(
                "Model drift monitoring",
                "PASS",
                f"Drift report contains {len(psi_values)} valid PSI values."
            )

# Model retraining criteria verification
retraining_md = read_text(
    DOCUMENTATION_DIR / "MODEL_RETRAINING_CRITERIA.md"
)

retraining_json_path = DOCUMENTATION_DIR / "MODEL_RETRAINING_CRITERIA.json"

try:
    with open(retraining_json_path, "r", encoding="utf-8") as file:
        retraining_json = json.load(file)

    valid_retraining_json = isinstance(retraining_json, (dict, list)) and bool(retraining_json)
except Exception:
    valid_retraining_json = False

required_terms = ["retrain", "threshold", "performance", "drift"]

retraining_doc_valid = (
    len(retraining_md.strip()) > 100
    and all(term in retraining_md.lower() for term in required_terms)
)

if retraining_doc_valid and valid_retraining_json:
    record("Retraining criteria", "PASS", "Retraining documentation and JSON are valid.")
else:
    record("Retraining criteria", "NOT VERIFIED", "Retraining documentation or JSON incomplete.")

# Model version documentation verification
version_path = DOCUMENTATION_DIR / "MODEL_VERSION_DOCUMENTATION.json"
version_inventory = read_csv(
    DEPLOYMENT_DIR / "model_version_inventory.csv"
)

try:
    with open(version_path, "r", encoding="utf-8") as file:
        version_data = json.load(file)

    valid_version_json = isinstance(version_data, (dict, list)) and bool(version_data)
except Exception:
    valid_version_json = False

if valid_version_json and version_inventory is not None and not version_inventory.empty:
    record(
        "Model version documentation",
        "PASS",
        f"Version JSON valid; inventory contains {len(version_inventory)} rows."
    )
else:
    record(
        "Model version documentation",
        "NOT VERIFIED",
        "Version documentation or inventory missing or invalid."
    )

# Final deployment maintenance checklist
final_report = pd.DataFrame(results)

final_report.to_csv(
    DEPLOYMENT_DIR / "final_deployment_maintenance_checklist.csv",
    index=False
)

display(final_report)
display(refresh_report)

passed = int((final_report["Status"] == "PASS").sum())
failed = int((final_report["Status"] == "FAIL").sum())
unverified = int((final_report["Status"] == "NOT VERIFIED").sum())

print("\nPASS:", passed, "/ 6")
print("FAIL:", failed, "/ 6")
print("NOT VERIFIED:", unverified, "/ 6")
print("Final report:", DEPLOYMENT_DIR / "final_deployment_maintenance_checklist.csv")

if passed == 6:
    print("ALL SIX VERIFICATIONS PASSED.")
else:
    print("FINAL COMPLETION NOT CONFIRMED.")

,Verification,Status,Details
0,Forecast reproducibility testing,PASS,20 checks passed.
1,Data refresh procedure,PASS,720 non-empty raw records; Raw date values val...
2,Model monitoring procedure,PASS,Monitoring metrics contain 7 rows; documentati...
3,Model drift monitoring,PASS,Drift report contains 1 valid PSI values.
4,Retraining criteria,PASS,Retraining documentation and JSON are valid.
5,Model version documentation,PASS,Version JSON valid; inventory contains 20 rows.


,Check,Status
0,Raw date values valid,PASS
1,Raw target values complete,PASS
2,Raw dates contain no duplicates,PASS
3,Processed dataset valid,PASS
4,Raw and processed record counts match,PASS
5,All raw and processed records match,PASS



PASS: 6 / 6
FAIL: 0 / 6
NOT VERIFIED: 0 / 6
Final report: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\deployment\final_deployment_maintenance_checklist.csv
ALL SIX VERIFICATIONS PASSED.
